In [ ]:
!pip install requests pandas

In [ ]:
import getpass

API_KEY = getpass.getpass("Enter your SerpApi API key: ")

Enter your SerpApi API key: ··········


In [ ]:
import requests
import pandas as pd
import getpass
import time

# ============================================================
# 1. CONFIGURATION
# ============================================================

API_KEY = getpass.getpass("Enter your SerpApi API key: ")

CITY = "Chennai"
STATE = "Tamil Nadu"
COUNTRY_CODE = "in"

# Google Hotels requires stay dates.
CHECK_IN = "2026-10-15"
CHECK_OUT = "2026-10-16"

# Collect up to 1000 unique hotels
TARGET_HOTELS = 1000

# Maximum number of review pages per hotel
MAX_REVIEW_PAGES = 3

OUTPUT_FILE = "chennai_hotel_dataset.csv"
BASE_URL = "https://serpapi.com/search.json"

# Search multiple areas to increase hotel coverage.
AREAS = [
    "Central Chennai",
    "T Nagar",
    "Nungambakkam",
    "Anna Nagar",
    "Adyar",
    "Besant Nagar",
    "Velachery",
    "Guindy",
    "OMR",
    "Sholinganallur",
    "Perungudi",
    "Thoraipakkam",
    "ECR",
    "Mylapore",
    "Alwarpet",
    "Royapettah",
    "Egmore",
    "Kilpauk",
    "Porur",
    "Vadapalani",
    "Koyambedu",
    "Chromepet",
    "Pallavaram",
    "Tambaram",
    "Guindy",
    "Mount Road",
    "Chennai Airport",
    "Perambur",
    "Ambattur",
    "Madhavaram",
    "Washermanpet",
    "Triplicane",
    "Royapuram",
    "Alandur",
    "Saidapet",
    "Kodambakkam",
    "Ashok Nagar",
    "KK Nagar",
    "Purasawalkam",
    "Chetpet",
    "Shenoy Nagar",
    "RA Puram",
    "Thiruvanmiyur",
    "Neelankarai",
    "Kottivakkam",
    "Medavakkam",
    "Pallikaranai",
    "Madipakkam"
]

# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def safe_request(params):
    params = params.copy()
    params["api_key"] = API_KEY

    try:
        response = requests.get(
            BASE_URL,
            params=params,
            timeout=60
        )

        if response.status_code != 200:
            print("API error:", response.status_code)
            print(response.text[:300])
            return {}

        data = response.json()

        if "error" in data:
            print("API error:", data["error"])
            return {}

        return data

    except requests.RequestException as e:
        print("Request failed:", e)
        return {}


def extract_price(hotel):
    rate = hotel.get("rate_per_night", {})

    if isinstance(rate, dict):
        return (
            rate.get("extracted_lowest")
            or rate.get("extracted_before_taxes_fees")
            or rate.get("lowest")
            or rate.get("before_taxes_fees")
        )

    return rate if isinstance(rate, (int, float)) else None


def extract_amenities(hotel):
    amenities = hotel.get("amenities", [])

    if isinstance(amenities, list):
        names = []

        for item in amenities:
            if isinstance(item, dict):
                name = item.get("name") or item.get("title")

                if name:
                    names.append(str(name))
            else:
                names.append(str(item))

        return " | ".join(names)

    return str(amenities) if amenities else None


def extract_location(hotel):
    address = hotel.get("address")

    if address:
        return address

    gps = hotel.get("gps_coordinates", {})

    if isinstance(gps, dict):
        lat = gps.get("latitude")
        lon = gps.get("longitude")

        if lat is not None and lon is not None:
            return f"Latitude: {lat}, Longitude: {lon}"

    return None


def extract_review_text(review):

    for key in ("snippet", "text"):

        value = review.get(key)

        if isinstance(value, str) and value.strip():
            return value.strip()

    return None


def extract_review_rating(review):

    rating = review.get("rating")

    if isinstance(rating, dict):
        return (
            rating.get("value")
            or rating.get("rating")
        )

    return rating


def make_hotel_row(hotel, area):

    return {
        "hotel_name": hotel.get("name"),

        "hotel_id": hotel.get("property_token"),

        "city": CITY,

        "location": extract_location(hotel),

        "price_per_night": extract_price(hotel),

        "hotel_category": hotel.get("hotel_class"),

        "customer_rating": hotel.get("overall_rating"),

        "review_count": hotel.get("reviews"),

        "amenities": extract_amenities(hotel),

        "search_area": area
    }


# ============================================================
# 3. SEARCH HOTELS ACROSS CHENNAI
# ============================================================

hotels_by_id = {}

for area in AREAS:

    if len(hotels_by_id) >= TARGET_HOTELS:
        break

    print(f"\nSearching: Hotels in {area}, {CITY}")

    next_page_token = None

    while len(hotels_by_id) < TARGET_HOTELS:

        params = {
            "engine": "google_hotels",

            "q": f"Hotels in {area}, {CITY}",

            "gl": COUNTRY_CODE,

            "hl": "en",

            "currency": "INR",

            "check_in_date": CHECK_IN,

            "check_out_date": CHECK_OUT
        }

        if next_page_token:

            params["next_page_token"] = next_page_token

        data = safe_request(params)

        if not data:
            break

        properties = data.get("properties", [])

        if not properties:
            break

        for hotel in properties:

            hotel_id = hotel.get("property_token")

            if not hotel_id:
                continue

            # Keep only one record per unique hotel.
            if hotel_id not in hotels_by_id:

                hotels_by_id[hotel_id] = {
                    **make_hotel_row(hotel, area),

                    "_raw_hotel": hotel
                }

            if len(hotels_by_id) >= TARGET_HOTELS:
                break

        print(
            "Unique Chennai hotels collected:",
            len(hotels_by_id)
        )

        next_page_token = (
            data.get("serpapi_pagination", {})
            .get("next_page_token")
        )

        if not next_page_token:
            break

        time.sleep(1)

    if len(hotels_by_id) >= TARGET_HOTELS:
        break


print("\n========================================")
print("Unique Chennai hotels collected:",
      len(hotels_by_id))
print("========================================")


# ============================================================
# 4. FETCH REVIEWS FOR EACH HOTEL
# ============================================================

all_rows = []

skipped_reviews = 0

hotel_items = list(hotels_by_id.values())


for hotel_num, hotel_data in enumerate(
    hotel_items,
    start=1
):

    hotel = hotel_data["_raw_hotel"]

    hotel_id = hotel.get("property_token")

    print(
        f"\n[{hotel_num}/{len(hotel_items)}] "
        f"{hotel.get('name')}"
    )

    hotel_details = {
        key: value
        for key, value in hotel_data.items()
        if key != "_raw_hotel"
    }

    next_review_token = None

    review_pages = 0

    valid_reviews = 0

    while review_pages < MAX_REVIEW_PAGES:

        params = {

            "engine": "google_hotels_reviews",

            "property_token": hotel_id,

            "hl": "en",

            "sort_by": 2
        }

        if next_review_token:

            params["next_page_token"] = (
                next_review_token
            )

        review_data = safe_request(params)

        if not review_data:
            break

        reviews = review_data.get(
            "reviews",
            []
        )

        if not reviews:
            break

        review_pages += 1

        for review in reviews:

            review_text = extract_review_text(
                review
            )

            if not review_text:

                skipped_reviews += 1

                continue

            row = hotel_details.copy()

            row.update({

                "review_text":
                    review_text,

                "individual_review_rating":
                    extract_review_rating(review),

                "review_date":
                    review.get("date"),

                "review_source":
                    review.get("source")
            })

            all_rows.append(row)

            valid_reviews += 1

        next_review_token = (
            review_data
            .get("serpapi_pagination", {})
            .get("next_page_token")
        )

        if not next_review_token:
            break

        time.sleep(1)

    print(
        "Reviews with text:",
        valid_reviews
    )

    time.sleep(1)


# ============================================================
# 5. SAVE THE COMBINED DATASET
# ============================================================

columns = [

    "hotel_name",

    "hotel_id",

    "city",

    "location",

    "price_per_night",

    "hotel_category",

    "customer_rating",

    "review_count",

    "amenities",

    "review_text",

    "individual_review_rating",

    "review_date",

    "review_source",

    "search_area"
]


df = pd.DataFrame(
    all_rows,
    columns=columns
)


if not df.empty:

    # Remove rows without review text.

    df = df[
        df["review_text"].notna()
        &
        df["review_text"]
        .astype(str)
        .str.strip()
        .ne("")
    ].copy()

    # Remove exact duplicate rows.

    df.drop_duplicates(
        inplace=True
    )


# ============================================================
# 6. SAVE CSV
# ============================================================

df.to_csv(

    OUTPUT_FILE,

    index=False,

    encoding="utf-8-sig"
)


# ============================================================
# 7. FINAL SUMMARY
# ============================================================

print("\n")
print("==============================================")
print("       CHENNAI HOTEL DATASET SUMMARY")
print("==============================================")

print(
    "Target hotels:",
    TARGET_HOTELS
)

print(
    "Unique hotels found:",
    len(hotels_by_id)
)

print(
    "Hotels with text-bearing reviews:",
    df["hotel_id"].nunique()
    if not df.empty
    else 0
)

print(
    "Review rows saved:",
    len(df)
)

print(
    "Reviews skipped for empty text:",
    skipped_reviews
)

print(
    "CSV saved:",
    OUTPUT_FILE
)

print("==============================================")


display(
    df.head(10)
)

Streaming output truncated to the last 5000 lines.
{
  "error": "Your account has run out of searches."
}
Reviews with text: 0

[289/1000] KLR GUEST HOUSE
API error: 429
{
  "error": "Your account has run out of searches."
}
Reviews with text: 0

[290/1000] Hotel O by OYO Tnagar Residency
API error: 429
{
  "error": "Your account has run out of searches."
}
Reviews with text: 0

[291/1000] OYO 12236 The Pearl
API error: 429
{
  "error": "Your account has run out of searches."
}
Reviews with text: 0

[292/1000] Hotel O Golden Tower
API error: 429
{
  "error": "Your account has run out of searches."
}
Reviews with text: 0

[293/1000] Metro Palace
API error: 429
{
  "error": "Your account has run out of searches."
}
Reviews with text: 0

[294/1000] Kuber Inn
API error: 429
{
  "error": "Your account has run out of searches."
}
Reviews with text: 0

[295/1000] Hotel Swagath
API error: 429
{
  "error": "Your account has run out of searches."
}
Reviews with text: 0

[296/1000] Kings Hotel - 

,hotel_name,hotel_id,city,location,price_per_night,hotel_category,customer_rating,review_count,amenities,review_text,individual_review_rating,review_date,review_source,search_area
0,The Hydel Park (Business Class Hotel) Chennai,ChkIqvXWlP-Y1ut1Gg0vZy8xMXJrcGRneTB5EAE,Chennai,"Latitude: 13.0819449, Longitude: 80.2704062",2828.0,None,4.7,3599,Breakfast ($) | Free Wi-Fi | Free parking | Ai...,Good Experience,4.0,6 days ago,Google,Central Chennai
1,The Hydel Park (Business Class Hotel) Chennai,ChkIqvXWlP-Y1ut1Gg0vZy8xMXJrcGRneTB5EAE,Chennai,"Latitude: 13.0819449, Longitude: 80.2704062",2828.0,None,4.7,3599,Breakfast ($) | Free Wi-Fi | Free parking | Ai...,Veryy good staff very nice I highly recommend ...,5.0,6 days ago,Google,Central Chennai
2,The Hydel Park (Business Class Hotel) Chennai,ChkIqvXWlP-Y1ut1Gg0vZy8xMXJrcGRneTB5EAE,Chennai,"Latitude: 13.0819449, Longitude: 80.2704062",2828.0,None,4.7,3599,Breakfast ($) | Free Wi-Fi | Free parking | Ai...,The rooms are very clean and well maintained a...,5.0,a week ago,Google,Central Chennai
3,The Hydel Park (Business Class Hotel) Chennai,ChkIqvXWlP-Y1ut1Gg0vZy8xMXJrcGRneTB5EAE,Chennai,"Latitude: 13.0819449, Longitude: 80.2704062",2828.0,None,4.7,3599,Breakfast ($) | Free Wi-Fi | Free parking | Ai...,Excellent Service & Professional Hospitality\n...,5.0,3 weeks ago,Google,Central Chennai
4,The Hydel Park (Business Class Hotel) Chennai,ChkIqvXWlP-Y1ut1Gg0vZy8xMXJrcGRneTB5EAE,Chennai,"Latitude: 13.0819449, Longitude: 80.2704062",2828.0,None,4.7,3599,Breakfast ($) | Free Wi-Fi | Free parking | Ai...,Very nice staff and clean and neat hotel,5.0,a month ago,Google,Central Chennai
5,The Hydel Park (Business Class Hotel) Chennai,ChkIqvXWlP-Y1ut1Gg0vZy8xMXJrcGRneTB5EAE,Chennai,"Latitude: 13.0819449, Longitude: 80.2704062",2828.0,None,4.7,3599,Breakfast ($) | Free Wi-Fi | Free parking | Ai...,"Best service by the staff, was very much impre...",5.0,a month ago,Google,Central Chennai
6,The Hydel Park (Business Class Hotel) Chennai,ChkIqvXWlP-Y1ut1Gg0vZy8xMXJrcGRneTB5EAE,Chennai,"Latitude: 13.0819449, Longitude: 80.2704062",2828.0,None,4.7,3599,Breakfast ($) | Free Wi-Fi | Free parking | Ai...,Good service and hotel located near by Chennai...,5.0,a month ago,Google,Central Chennai
7,The Hydel Park (Business Class Hotel) Chennai,ChkIqvXWlP-Y1ut1Gg0vZy8xMXJrcGRneTB5EAE,Chennai,"Latitude: 13.0819449, Longitude: 80.2704062",2828.0,None,4.7,3599,Breakfast ($) | Free Wi-Fi | Free parking | Ai...,I made this reservation with Hydel Park nearly...,1.0,2 months ago,Google,Central Chennai
8,The Hydel Park (Business Class Hotel) Chennai,ChkIqvXWlP-Y1ut1Gg0vZy8xMXJrcGRneTB5EAE,Chennai,"Latitude: 13.0819449, Longitude: 80.2704062",2828.0,None,4.7,3599,Breakfast ($) | Free Wi-Fi | Free parking | Ai...,Rooms and washrooms were clean and the service...,5.0,2 months ago,Google,Central Chennai
9,The Hydel Park (Business Class Hotel) Chennai,ChkIqvXWlP-Y1ut1Gg0vZy8xMXJrcGRneTB5EAE,Chennai,"Latitude: 13.0819449, Longitude: 80.2704062",2828.0,None,4.7,3599,Breakfast ($) | Free Wi-Fi | Free parking | Ai...,Good experience with the hydel park,5.0,2 months ago,Google,Central Chennai


In [ ]:
from google.colab import files
files.download("chennai_hotel_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import requests
import pandas as pd
import getpass
import time

# ============================================================
# 1. CONFIGURATION
# ============================================================

API_KEY = getpass.getpass("Enter your SerpApi API key: ")

CITY = "Pune"
STATE = "Maharashtra"
COUNTRY_CODE = "in"

# Google Hotels requires stay dates.
CHECK_IN = "2026-10-15"
CHECK_OUT = "2026-10-16"

# Collect up to 500 unique hotels
TARGET_HOTELS = 500

# Maximum number of review pages per hotel
MAX_REVIEW_PAGES = 3

OUTPUT_FILE = "pune_hotel_dataset.csv"
BASE_URL = "https://serpapi.com/search.json"

# Search multiple areas of Pune to increase coverage.
AREAS = [
    # Central Pune
    "Central Pune",
    "Shivajinagar",
    "Deccan Gymkhana",
    "Camp",
    "Swargate",
    "Shaniwar Peth",
    "Sadashiv Peth",
    "Budhwar Peth",

    # Koregaon Park / East Pune
    "Koregaon Park",
    "Kalyani Nagar",
    "Viman Nagar",
    "Kharadi",
    "Mundhwa",
    "Magarpatta",

    # Hinjewadi / West Pune
    "Hinjewadi",
    "Wakad",
    "Baner",
    "Balewadi",
    "Aundh",
    "Pashan",
    "Bavdhan",
    "Sus",

    # South Pune
    "Katraj",
    "Kondhwa",
    "NIBM",
    "Wanowrie",
    "Hadapsar",
    "Fursungi",
    "Bibwewadi",
    "Dhankawadi",

    # North Pune
    "Kothrud",
    "Warje",
    "Bopodi",
    "Khadki",
    "Akurdi",
    "Pimpri",
    "Chinchwad",
    "Bhosari",
    "Nigdi",

    # Pune Airport / nearby
    "Pune Airport",
    "Lohegaon",
    "Dhanori",
    "Vishrantwadi",

    # Additional areas
    "Yerawada",
    "Vishal Nagar",
    "Pimple Saudagar",
    "Pimple Gurav",
    "Ravet",
    "Talegaon"
]


# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def safe_request(params):
    params = params.copy()
    params["api_key"] = API_KEY

    try:
        response = requests.get(
            BASE_URL,
            params=params,
            timeout=60
        )

        if response.status_code != 200:
            print("API error:", response.status_code)
            print(response.text[:300])
            return {}

        data = response.json()

        if "error" in data:
            print("API error:", data["error"])
            return {}

        return data

    except requests.RequestException as e:
        print("Request failed:", e)
        return {}


def extract_price(hotel):
    rate = hotel.get("rate_per_night", {})

    if isinstance(rate, dict):
        return (
            rate.get("extracted_lowest")
            or rate.get("extracted_before_taxes_fees")
            or rate.get("lowest")
            or rate.get("before_taxes_fees")
        )

    return rate if isinstance(rate, (int, float)) else None


def extract_amenities(hotel):
    amenities = hotel.get("amenities", [])

    if isinstance(amenities, list):
        names = []

        for item in amenities:
            if isinstance(item, dict):
                name = item.get("name") or item.get("title")

                if name:
                    names.append(str(name))
            else:
                names.append(str(item))

        return " | ".join(names)

    return str(amenities) if amenities else None


def extract_location(hotel):
    address = hotel.get("address")

    if address:
        return address

    gps = hotel.get("gps_coordinates", {})

    if isinstance(gps, dict):
        lat = gps.get("latitude")
        lon = gps.get("longitude")

        if lat is not None and lon is not None:
            return f"Latitude: {lat}, Longitude: {lon}"

    return None


def extract_review_text(review):

    for key in ("snippet", "text"):

        value = review.get(key)

        if isinstance(value, str) and value.strip():
            return value.strip()

    return None


def extract_review_rating(review):

    rating = review.get("rating")

    if isinstance(rating, dict):
        return (
            rating.get("value")
            or rating.get("rating")
        )

    return rating


def make_hotel_row(hotel, area):

    return {
        "hotel_name": hotel.get("name"),

        "hotel_id": hotel.get("property_token"),

        "city": CITY,

        "location": extract_location(hotel),

        "price_per_night": extract_price(hotel),

        "hotel_category": hotel.get("hotel_class"),

        "customer_rating": hotel.get("overall_rating"),

        "review_count": hotel.get("reviews"),

        "amenities": extract_amenities(hotel),

        "search_area": area
    }


# ============================================================
# 3. SEARCH HOTELS ACROSS PUNE
# ============================================================

hotels_by_id = {}

for area in AREAS:

    if len(hotels_by_id) >= TARGET_HOTELS:
        break

    print(f"\nSearching: Hotels in {area}, {CITY}")

    next_page_token = None

    while len(hotels_by_id) < TARGET_HOTELS:

        params = {
            "engine": "google_hotels",

            "q": f"Hotels in {area}, {CITY}",

            "gl": COUNTRY_CODE,

            "hl": "en",

            "currency": "INR",

            "check_in_date": CHECK_IN,

            "check_out_date": CHECK_OUT
        }

        if next_page_token:

            params["next_page_token"] = next_page_token

        data = safe_request(params)

        if not data:
            break

        properties = data.get("properties", [])

        if not properties:
            break

        for hotel in properties:

            hotel_id = hotel.get("property_token")

            if not hotel_id:
                continue

            # Keep only one record per unique hotel.
            if hotel_id not in hotels_by_id:

                hotels_by_id[hotel_id] = {
                    **make_hotel_row(hotel, area),

                    "_raw_hotel": hotel
                }

            if len(hotels_by_id) >= TARGET_HOTELS:
                break

        print(
            "Unique Pune hotels collected:",
            len(hotels_by_id)
        )

        next_page_token = (
            data.get("serpapi_pagination", {})
            .get("next_page_token")
        )

        if not next_page_token:
            break

        time.sleep(1)

    if len(hotels_by_id) >= TARGET_HOTELS:
        break


print("\n========================================")
print(
    "Unique Pune hotels collected:",
    len(hotels_by_id)
)
print("========================================")


# ============================================================
# 4. FETCH REVIEWS FOR EACH HOTEL
# ============================================================

all_rows = []

skipped_reviews = 0

hotel_items = list(hotels_by_id.values())


for hotel_num, hotel_data in enumerate(
    hotel_items,
    start=1
):

    hotel = hotel_data["_raw_hotel"]

    hotel_id = hotel.get("property_token")

    print(
        f"\n[{hotel_num}/{len(hotel_items)}] "
        f"{hotel.get('name')}"
    )

    hotel_details = {
        key: value
        for key, value in hotel_data.items()
        if key != "_raw_hotel"
    }

    next_review_token = None

    review_pages = 0

    valid_reviews = 0

    while review_pages < MAX_REVIEW_PAGES:

        params = {

            "engine": "google_hotels_reviews",

            "property_token": hotel_id,

            "hl": "en",

            "sort_by": 2
        }

        if next_review_token:

            params["next_page_token"] = (
                next_review_token
            )

        review_data = safe_request(params)

        if not review_data:
            break

        reviews = review_data.get(
            "reviews",
            []
        )

        if not reviews:
            break

        review_pages += 1

        for review in reviews:

            review_text = extract_review_text(
                review
            )

            if not review_text:

                skipped_reviews += 1

                continue

            row = hotel_details.copy()

            row.update({

                "review_text":
                    review_text,

                "individual_review_rating":
                    extract_review_rating(review),

                "review_date":
                    review.get("date"),

                "review_source":
                    review.get("source")
            })

            all_rows.append(row)

            valid_reviews += 1

        next_review_token = (
            review_data
            .get("serpapi_pagination", {})
            .get("next_page_token")
        )

        if not next_review_token:
            break

        time.sleep(1)

    print(
        "Reviews with text:",
        valid_reviews
    )

    time.sleep(1)


# ============================================================
# 5. SAVE THE COMBINED DATASET
# ============================================================

columns = [

    "hotel_name",

    "hotel_id",

    "city",

    "location",

    "price_per_night",

    "hotel_category",

    "customer_rating",

    "review_count",

    "amenities",

    "review_text",

    "individual_review_rating",

    "review_date",

    "review_source",

    "search_area"
]


df = pd.DataFrame(
    all_rows,
    columns=columns
)


if not df.empty:

    # Remove rows without review text.

    df = df[
        df["review_text"].notna()
        &
        df["review_text"]
        .astype(str)
        .str.strip()
        .ne("")
    ].copy()

    # Remove exact duplicate rows.

    df.drop_duplicates(
        inplace=True
    )


# ============================================================
# 6. SAVE CSV
# ============================================================

df.to_csv(

    OUTPUT_FILE,

    index=False,

    encoding="utf-8-sig"
)


# ============================================================
# 7. FINAL SUMMARY
# ============================================================

print("\n")
print("==============================================")
print("        PUNE HOTEL DATASET SUMMARY")
print("==============================================")

print(
    "Target hotels:",
    TARGET_HOTELS
)

print(
    "Unique hotels found:",
    len(hotels_by_id)
)

print(
    "Hotels with text-bearing reviews:",
    df["hotel_id"].nunique()
    if not df.empty
    else 0
)

print(
    "Review rows saved:",
    len(df)
)

print(
    "Reviews skipped for empty text:",
    skipped_reviews
)

print(
    "CSV saved:",
    OUTPUT_FILE
)

print("==============================================")


display(
    df.head(10)
)


Searching: Hotels in Central Pune, Pune
Unique Pune hotels collected: 20
Unique Pune hotels collected: 37
Unique Pune hotels collected: 55
Unique Pune hotels collected: 72
Unique Pune hotels collected: 87
Unique Pune hotels collected: 101
Unique Pune hotels collected: 121
Unique Pune hotels collected: 135
Unique Pune hotels collected: 139
Unique Pune hotels collected: 151
Unique Pune hotels collected: 166

Searching: Hotels in Shivajinagar, Pune
Unique Pune hotels collected: 185
Unique Pune hotels collected: 199
Unique Pune hotels collected: 212
Unique Pune hotels collected: 225
Unique Pune hotels collected: 245
Unique Pune hotels collected: 264
Unique Pune hotels collected: 282
Unique Pune hotels collected: 292
Unique Pune hotels collected: 295

Searching: Hotels in Deccan Gymkhana, Pune
Unique Pune hotels collected: 297
Unique Pune hotels collected: 302
Unique Pune hotels collected: 308
Unique Pune hotels collected: 312
Unique Pune hotels collected: 314
Unique Pune hotels collected:

,hotel_name,hotel_id,city,location,price_per_night,hotel_category,customer_rating,review_count,amenities,review_text,individual_review_rating,review_date,review_source,search_area
0,"The Central Park Hotel, Pune",ChoIveHZrrPT_bDRARoNL2cvMTFiYnJnczh3YxAB,Pune,"Latitude: 18.5271764, Longitude: 73.8785874",4588.0,4-star hotel,4.1,4693,Free breakfast | Wi-Fi | Free parking | Outdoo...,Good location with peaceful surroundings. Unfo...,3,5 days ago,Google,Central Pune
1,"The Central Park Hotel, Pune",ChoIveHZrrPT_bDRARoNL2cvMTFiYnJnczh3YxAB,Pune,"Latitude: 18.5271764, Longitude: 73.8785874",4588.0,4-star hotel,4.1,4693,Free breakfast | Wi-Fi | Free parking | Outdoo...,Ankit singh,5,2 weeks ago,Google,Central Pune
2,"The Central Park Hotel, Pune",ChoIveHZrrPT_bDRARoNL2cvMTFiYnJnczh3YxAB,Pune,"Latitude: 18.5271764, Longitude: 73.8785874",4588.0,4-star hotel,4.1,4693,Free breakfast | Wi-Fi | Free parking | Outdoo...,"Hotel Feedback – Central Park, Pune||We had bo...",5,2 weeks ago,Tripadvisor,Central Pune
3,"The Central Park Hotel, Pune",ChoIveHZrrPT_bDRARoNL2cvMTFiYnJnczh3YxAB,Pune,"Latitude: 18.5271764, Longitude: 73.8785874",4588.0,4-star hotel,4.1,4693,Free breakfast | Wi-Fi | Free parking | Outdoo...,"Great stay, excellent staff and very clean roo...",5,2 weeks ago,Tripadvisor,Central Pune
4,"The Central Park Hotel, Pune",ChoIveHZrrPT_bDRARoNL2cvMTFiYnJnczh3YxAB,Pune,"Latitude: 18.5271764, Longitude: 73.8785874",4588.0,4-star hotel,4.1,4693,Free breakfast | Wi-Fi | Free parking | Outdoo...,"Great place, great food, amazing hospitality. ...",5,2 weeks ago,Tripadvisor,Central Pune
5,"The Central Park Hotel, Pune",ChoIveHZrrPT_bDRARoNL2cvMTFiYnJnczh3YxAB,Pune,"Latitude: 18.5271764, Longitude: 73.8785874",4588.0,4-star hotel,4.1,4693,Free breakfast | Wi-Fi | Free parking | Outdoo...,Located in city centre|Wonderful atmosphere n ...,5,3 weeks ago,Tripadvisor,Central Pune
6,"The Central Park Hotel, Pune",ChoIveHZrrPT_bDRARoNL2cvMTFiYnJnczh3YxAB,Pune,"Latitude: 18.5271764, Longitude: 73.8785874",4588.0,4-star hotel,4.1,4693,Free breakfast | Wi-Fi | Free parking | Outdoo...,1 great hospitality |2 very good location|3 go...,5,3 weeks ago,Tripadvisor,Central Pune
7,"The Central Park Hotel, Pune",ChoIveHZrrPT_bDRARoNL2cvMTFiYnJnczh3YxAB,Pune,"Latitude: 18.5271764, Longitude: 73.8785874",4588.0,4-star hotel,4.1,4693,Free breakfast | Wi-Fi | Free parking | Outdoo...,Great experience overall. The location was con...,5,3 weeks ago,Tripadvisor,Central Pune
8,"The Central Park Hotel, Pune",ChoIveHZrrPT_bDRARoNL2cvMTFiYnJnczh3YxAB,Pune,"Latitude: 18.5271764, Longitude: 73.8785874",4588.0,4-star hotel,4.1,4693,Free breakfast | Wi-Fi | Free parking | Outdoo...,"Location is Good, ample of parking space and e...",5,3 weeks ago,Tripadvisor,Central Pune
9,"The Central Park Hotel, Pune",ChoIveHZrrPT_bDRARoNL2cvMTFiYnJnczh3YxAB,Pune,"Latitude: 18.5271764, Longitude: 73.8785874",4588.0,4-star hotel,4.1,4693,Free breakfast | Wi-Fi | Free parking | Outdoo...,Smooth check in and out . Very courteous staff...,5,3 weeks ago,Tripadvisor,Central Pune


In [ ]:
from google.colab import files
files.download("pune_hotel_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import requests
import pandas as pd
import getpass
import time

# ============================================================
# 1. CONFIGURATION
# ============================================================

API_KEY = getpass.getpass("Enter your SerpApi API key: ")

CITY = "Andhra Pradesh"
STATE = "Andhra Pradesh"
COUNTRY_CODE = "in"

# Google Hotels requires stay dates.
CHECK_IN = "2026-10-15"
CHECK_OUT = "2026-10-16"

# Collect up to 500 unique hotels
TARGET_HOTELS = 500

# Maximum number of review pages per hotel
MAX_REVIEW_PAGES = 3

OUTPUT_FILE = "andhra_pradesh_hotel_dataset.csv"
BASE_URL = "https://serpapi.com/search.json"


# ============================================================
# ANDHRA PRADESH LOCATIONS
# ============================================================

AREAS = [

    # --------------------------------------------------------
    # VISAKHAPATNAM REGION
    # --------------------------------------------------------

    "Visakhapatnam",
    "Vizag",
    "Dwaraka Nagar Visakhapatnam",
    "MVP Colony Visakhapatnam",
    "Siripuram Visakhapatnam",
    "Madhurawada Visakhapatnam",
    "Gajuwaka Visakhapatnam",
    "Rushikonda Visakhapatnam",
    "Beach Road Visakhapatnam",
    "RK Beach Visakhapatnam",
    "Waltair Visakhapatnam",
    "Airport Visakhapatnam",

    # --------------------------------------------------------
    # VIJAYAWADA REGION
    # --------------------------------------------------------

    "Vijayawada",
    "Benz Circle Vijayawada",
    "Governorpet Vijayawada",
    "Moghalrajpuram Vijayawada",
    "Labbipet Vijayawada",
    "Bhavanipuram Vijayawada",
    "Poranki Vijayawada",
    "Gollapudi Vijayawada",

    # --------------------------------------------------------
    # TIRUPATI REGION
    # --------------------------------------------------------

    "Tirupati",
    "Tirumala",
    "Tiruchanur",
    "Renigunta",
    "Tirupati Railway Station",
    "Tirupati Airport",

    # --------------------------------------------------------
    # GUNTUR REGION
    # --------------------------------------------------------

    "Guntur",
    "Brodipet Guntur",
    "Arundelpet Guntur",
    "Lakshmipuram Guntur",
    "Amaravati",
    "Mangalagiri",

    # --------------------------------------------------------
    # NELLORE REGION
    # --------------------------------------------------------

    "Nellore",
    "Nellore City",
    "Magunta Layout Nellore",
    "Trunk Road Nellore",
    "Kovur",

    # --------------------------------------------------------
    # KAKINADA REGION
    # --------------------------------------------------------

    "Kakinada",
    "Kakinada Beach",
    "Sarpavaram Kakinada",
    "Jagannaickpur Kakinada",

    # --------------------------------------------------------
    # RAJAHMUNDRY REGION
    # --------------------------------------------------------

    "Rajahmundry",
    "Rajamahendravaram",
    "Danavaipeta Rajahmundry",
    "Morampudi Rajahmundry",
    "Dowleswaram",

    # --------------------------------------------------------
    # ANANTAPUR REGION
    # --------------------------------------------------------

    "Anantapur",
    "Anantapur City",
    "Rudrampeta Anantapur",

    # --------------------------------------------------------
    # KURNOOL REGION
    # --------------------------------------------------------

    "Kurnool",
    "Kurnool City",
    "Budhawarpet Kurnool",
    "Nandyal",

    # --------------------------------------------------------
    # KADAPA REGION
    # --------------------------------------------------------

    "Kadapa",
    "YSR Kadapa",
    "Proddatur",
    "Rajampet",

    # --------------------------------------------------------
    # SRIKAKULAM REGION
    # --------------------------------------------------------

    "Srikakulam",
    "Srikakulam City",
    "Arasavalli",

    # --------------------------------------------------------
    # VIZIANAGARAM REGION
    # --------------------------------------------------------

    "Vizianagaram",
    "Vizianagaram City",

    # --------------------------------------------------------
    # ELURU REGION
    # --------------------------------------------------------

    "Eluru",
    "Eluru City",

    # --------------------------------------------------------
    # MACHILIPATNAM REGION
    # --------------------------------------------------------

    "Machilipatnam",
    "Machilipatnam Beach",

    # --------------------------------------------------------
    # BHIMAVARAM REGION
    # --------------------------------------------------------

    "Bhimavaram",
    "Bhimavaram City",

    # --------------------------------------------------------
    # ONGOLE REGION
    # --------------------------------------------------------

    "Ongole",
    "Ongole City",

    # --------------------------------------------------------
    # COASTAL / TOURISM
    # --------------------------------------------------------

    "Bapatla",
    "Bapatla Beach",
    "Suryalanka Beach",
    "Chirala",
    "Vodarevu Beach",
    "Kondapalli",

    # --------------------------------------------------------
    # ARTS / HERITAGE / PILGRIMAGE
    # --------------------------------------------------------

    "Lepakshi",
    "Srisailam",
    "Ahobilam",
    "Simhachalam",
    "Araku Valley",
    "Lambasingi",
    "Papikondalu",
    "Dwaraka Tirumala"
]


# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def safe_request(params):

    params = params.copy()
    params["api_key"] = API_KEY

    try:

        response = requests.get(
            BASE_URL,
            params=params,
            timeout=60
        )

        if response.status_code != 200:

            print(
                "API error:",
                response.status_code
            )

            print(
                response.text[:300]
            )

            return {}

        data = response.json()

        if "error" in data:

            print(
                "API error:",
                data["error"]
            )

            return {}

        return data

    except requests.RequestException as e:

        print(
            "Request failed:",
            e
        )

        return {}


def extract_price(hotel):

    rate = hotel.get(
        "rate_per_night",
        {}
    )

    if isinstance(rate, dict):

        return (

            rate.get(
                "extracted_lowest"
            )

            or

            rate.get(
                "extracted_before_taxes_fees"
            )

            or

            rate.get(
                "lowest"
            )

            or

            rate.get(
                "before_taxes_fees"
            )
        )

    return (
        rate
        if isinstance(
            rate,
            (int, float)
        )
        else None
    )


def extract_amenities(hotel):

    amenities = hotel.get(
        "amenities",
        []
    )

    if isinstance(
        amenities,
        list
    ):

        names = []

        for item in amenities:

            if isinstance(
                item,
                dict
            ):

                name = (
                    item.get("name")
                    or
                    item.get("title")
                )

                if name:

                    names.append(
                        str(name)
                    )

            else:

                names.append(
                    str(item)
                )

        return " | ".join(names)

    return (
        str(amenities)
        if amenities
        else None
    )


def extract_location(hotel):

    address = hotel.get(
        "address"
    )

    if address:

        return address

    gps = hotel.get(
        "gps_coordinates",
        {}
    )

    if isinstance(
        gps,
        dict
    ):

        lat = gps.get(
            "latitude"
        )

        lon = gps.get(
            "longitude"
        )

        if (
            lat is not None
            and
            lon is not None
        ):

            return (
                f"Latitude: {lat}, "
                f"Longitude: {lon}"
            )

    return None


def extract_review_text(review):

    for key in (
        "snippet",
        "text"
    ):

        value = review.get(
            key
        )

        if (
            isinstance(
                value,
                str
            )
            and
            value.strip()
        ):

            return value.strip()

    return None


def extract_review_rating(review):

    rating = review.get(
        "rating"
    )

    if isinstance(
        rating,
        dict
    ):

        return (
            rating.get(
                "value"
            )

            or

            rating.get(
                "rating"
            )
        )

    return rating


def make_hotel_row(
    hotel,
    area
):

    return {

        "hotel_name":
            hotel.get("name"),

        "hotel_id":
            hotel.get(
                "property_token"
            ),

        "state":
            STATE,

        "location":
            extract_location(
                hotel
            ),

        "price_per_night":
            extract_price(
                hotel
            ),

        "hotel_category":
            hotel.get(
                "hotel_class"
            ),

        "customer_rating":
            hotel.get(
                "overall_rating"
            ),

        "review_count":
            hotel.get(
                "reviews"
            ),

        "amenities":
            extract_amenities(
                hotel
            ),

        "search_area":
            area
    }


# ============================================================
# 3. SEARCH HOTELS ACROSS ANDHRA PRADESH
# ============================================================

hotels_by_id = {}


for area in AREAS:

    if (
        len(hotels_by_id)
        >= TARGET_HOTELS
    ):

        break

    print(
        f"\nSearching: Hotels in "
        f"{area}, Andhra Pradesh"
    )

    next_page_token = None


    while (
        len(hotels_by_id)
        < TARGET_HOTELS
    ):

        params = {

            "engine":
                "google_hotels",

            "q":
                f"Hotels in "
                f"{area}, Andhra Pradesh",

            "gl":
                COUNTRY_CODE,

            "hl":
                "en",

            "currency":
                "INR",

            "check_in_date":
                CHECK_IN,

            "check_out_date":
                CHECK_OUT
        }


        if next_page_token:

            params[
                "next_page_token"
            ] = next_page_token


        data = safe_request(
            params
        )


        if not data:

            break


        properties = data.get(
            "properties",
            []
        )


        if not properties:

            break


        for hotel in properties:

            hotel_id = hotel.get(
                "property_token"
            )


            if not hotel_id:

                continue


            # Keep only one record
            # per unique hotel.

            if (
                hotel_id
                not in hotels_by_id
            ):

                hotels_by_id[
                    hotel_id
                ] = {

                    **make_hotel_row(
                        hotel,
                        area
                    ),

                    "_raw_hotel":
                        hotel
                }


            if (
                len(hotels_by_id)
                >= TARGET_HOTELS
            ):

                break


        print(
            "Unique Andhra Pradesh "
            "hotels collected:",
            len(hotels_by_id)
        )


        next_page_token = (

            data
            .get(
                "serpapi_pagination",
                {}
            )
            .get(
                "next_page_token"
            )
        )


        if not next_page_token:

            break


        time.sleep(1)


    if (
        len(hotels_by_id)
        >= TARGET_HOTELS
    ):

        break


print(
    "\n========================================"
)

print(
    "Unique Andhra Pradesh hotels collected:",
    len(hotels_by_id)
)

print(
    "========================================"
)


# ============================================================
# 4. FETCH REVIEWS FOR EACH HOTEL
# ============================================================

all_rows = []

skipped_reviews = 0

hotel_items = list(
    hotels_by_id.values()
)


for hotel_num, hotel_data in enumerate(
    hotel_items,
    start=1
):

    hotel = hotel_data[
        "_raw_hotel"
    ]

    hotel_id = hotel.get(
        "property_token"
    )


    print(
        f"\n[{hotel_num}/"
        f"{len(hotel_items)}] "
        f"{hotel.get('name')}"
    )


    hotel_details = {

        key: value

        for key, value
        in hotel_data.items()

        if key != "_raw_hotel"
    }


    next_review_token = None

    review_pages = 0

    valid_reviews = 0


    while (
        review_pages
        < MAX_REVIEW_PAGES
    ):

        params = {

            "engine":
                "google_hotels_reviews",

            "property_token":
                hotel_id,

            "hl":
                "en",

            "sort_by":
                2
        }


        if next_review_token:

            params[
                "next_page_token"
            ] = next_review_token


        review_data = safe_request(
            params
        )


        if not review_data:

            break


        reviews = review_data.get(
            "reviews",
            []
        )


        if not reviews:

            break


        review_pages += 1


        for review in reviews:

            review_text = (
                extract_review_text(
                    review
                )
            )


            if not review_text:

                skipped_reviews += 1

                continue


            row = (
                hotel_details.copy()
            )


            row.update({

                "review_text":
                    review_text,

                "individual_review_rating":
                    extract_review_rating(
                        review
                    ),

                "review_date":
                    review.get(
                        "date"
                    ),

                "review_source":
                    review.get(
                        "source"
                    )
            })


            all_rows.append(
                row
            )

            valid_reviews += 1


        next_review_token = (

            review_data

            .get(
                "serpapi_pagination",
                {}
            )

            .get(
                "next_page_token"
            )
        )


        if not next_review_token:

            break


        time.sleep(1)


    print(
        "Reviews with text:",
        valid_reviews
    )


    time.sleep(1)


# ============================================================
# 5. SAVE THE COMBINED DATASET
# ============================================================

columns = [

    "hotel_name",

    "hotel_id",

    "state",

    "location",

    "price_per_night",

    "hotel_category",

    "customer_rating",

    "review_count",

    "amenities",

    "review_text",

    "individual_review_rating",

    "review_date",

    "review_source",

    "search_area"
]


df = pd.DataFrame(
    all_rows,
    columns=columns
)


if not df.empty:

    # Remove rows without review text.

    df = df[
        df["review_text"]
        .notna()

        &

        df["review_text"]
        .astype(str)
        .str.strip()
        .ne("")
    ].copy()


    # Remove exact duplicate rows.

    df.drop_duplicates(
        inplace=True
    )


# ============================================================
# 6. SAVE CSV
# ============================================================

df.to_csv(

    OUTPUT_FILE,

    index=False,

    encoding="utf-8-sig"
)


# ============================================================
# 7. FINAL SUMMARY
# ============================================================

print("\n")

print(
    "=============================================="
)

print(
    "   ANDHRA PRADESH HOTEL DATASET SUMMARY"
)

print(
    "=============================================="
)


print(
    "Target hotels:",
    TARGET_HOTELS
)


print(
    "Unique hotels found:",
    len(hotels_by_id)
)


print(
    "Hotels with text-bearing reviews:",
    df["hotel_id"].nunique()
    if not df.empty
    else 0
)


print(
    "Review rows saved:",
    len(df)
)


print(
    "Reviews skipped for empty text:",
    skipped_reviews
)


print(
    "CSV saved:",
    OUTPUT_FILE
)


print(
    "=============================================="
)


display(
    df.head(10)
)

Enter your SerpApi API key: ··········

Searching: Hotels in Visakhapatnam, Andhra Pradesh
Unique Andhra Pradesh hotels collected: 20
Unique Andhra Pradesh hotels collected: 37
Unique Andhra Pradesh hotels collected: 55
Unique Andhra Pradesh hotels collected: 65
Unique Andhra Pradesh hotels collected: 78
Unique Andhra Pradesh hotels collected: 96
Unique Andhra Pradesh hotels collected: 113
Unique Andhra Pradesh hotels collected: 129
Unique Andhra Pradesh hotels collected: 138
Unique Andhra Pradesh hotels collected: 153
Unique Andhra Pradesh hotels collected: 169
Unique Andhra Pradesh hotels collected: 185
Unique Andhra Pradesh hotels collected: 199
Unique Andhra Pradesh hotels collected: 210
Unique Andhra Pradesh hotels collected: 226
Unique Andhra Pradesh hotels collected: 238
Unique Andhra Pradesh hotels collected: 256
Unique Andhra Pradesh hotels collected: 270
Unique Andhra Pradesh hotels collected: 279
Unique Andhra Pradesh hotels collected: 291
Unique Andhra Pradesh hotels collec

,hotel_name,hotel_id,state,location,price_per_night,hotel_category,customer_rating,review_count,amenities,review_text,individual_review_rating,review_date,review_source,search_area
0,Hotel Rockdale,ChoI6PWk9reL6YP3ARoNL2cvMTFjazgyaHJmeRAB,Andhra Pradesh,"Latitude: 17.716386, Longitude: 83.308313",1981.0,3-star hotel,4.4,11853,Free breakfast | Free Wi-Fi | Free parking | A...,It was great experience having food at rockdal...,5,6 hours ago,Google,Visakhapatnam
1,Hotel Rockdale,ChoI6PWk9reL6YP3ARoNL2cvMTFjazgyaHJmeRAB,Andhra Pradesh,"Latitude: 17.716386, Longitude: 83.308313",1981.0,3-star hotel,4.4,11853,Free breakfast | Free Wi-Fi | Free parking | A...,I had a wonderful stay at Hotel Rockdale and r...,5,8 hours ago,Google,Visakhapatnam
2,Hotel Rockdale,ChoI6PWk9reL6YP3ARoNL2cvMTFjazgyaHJmeRAB,Andhra Pradesh,"Latitude: 17.716386, Longitude: 83.308313",1981.0,3-star hotel,4.4,11853,Free breakfast | Free Wi-Fi | Free parking | A...,Exceptional stay,5,a day ago,Google,Visakhapatnam
3,Hotel Rockdale,ChoI6PWk9reL6YP3ARoNL2cvMTFjazgyaHJmeRAB,Andhra Pradesh,"Latitude: 17.716386, Longitude: 83.308313",1981.0,3-star hotel,4.4,11853,Free breakfast | Free Wi-Fi | Free parking | A...,"Front office is very nice, helpful. Room clean...",3,a day ago,Google,Visakhapatnam
4,Hotel Rockdale,ChoI6PWk9reL6YP3ARoNL2cvMTFjazgyaHJmeRAB,Andhra Pradesh,"Latitude: 17.716386, Longitude: 83.308313",1981.0,3-star hotel,4.4,11853,Free breakfast | Free Wi-Fi | Free parking | A...,I enjoyed food and hospitality. Ambiance is fi...,5,Edited 2 days ago,Google,Visakhapatnam
5,Hotel Rockdale,ChoI6PWk9reL6YP3ARoNL2cvMTFjazgyaHJmeRAB,Andhra Pradesh,"Latitude: 17.716386, Longitude: 83.308313",1981.0,3-star hotel,4.4,11853,Free breakfast | Free Wi-Fi | Free parking | A...,Good,5,2 days ago,Google,Visakhapatnam
6,Hotel Rockdale,ChoI6PWk9reL6YP3ARoNL2cvMTFjazgyaHJmeRAB,Andhra Pradesh,"Latitude: 17.716386, Longitude: 83.308313",1981.0,3-star hotel,4.4,11853,Free breakfast | Free Wi-Fi | Free parking | A...,Not worth the money they charge. I dont recoomed.,2,3 days ago,Google,Visakhapatnam
7,Hotel Rockdale,ChoI6PWk9reL6YP3ARoNL2cvMTFjazgyaHJmeRAB,Andhra Pradesh,"Latitude: 17.716386, Longitude: 83.308313",1981.0,3-star hotel,4.4,11853,Free breakfast | Free Wi-Fi | Free parking | A...,We stayed at Rockdale for two nights in Deluxe...,5,3 days ago,Google,Visakhapatnam
8,Hotel Rockdale,ChoI6PWk9reL6YP3ARoNL2cvMTFjazgyaHJmeRAB,Andhra Pradesh,"Latitude: 17.716386, Longitude: 83.308313",1981.0,3-star hotel,4.4,11853,Free breakfast | Free Wi-Fi | Free parking | A...,Very good for stay and also good hospitality,5,3 days ago,Google,Visakhapatnam
9,Hotel Rockdale,ChoI6PWk9reL6YP3ARoNL2cvMTFjazgyaHJmeRAB,Andhra Pradesh,"Latitude: 17.716386, Longitude: 83.308313",1981.0,3-star hotel,4.4,11853,Free breakfast | Free Wi-Fi | Free parking | A...,Booked and stayed in superior room\nEverything...,5,3 days ago,Google,Visakhapatnam


In [ ]:
from google.colab import files
files.download("andhra_pradesh_hotel_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import time
import requests
import pandas as pd

# ============================================================
# CONFIGURATION
# ============================================================

API_KEY = input("Enter your SerpApi API key: ").strip()

OUTPUT_FILE = "chennai_hotel_dataset.csv"
CHECKPOINT_FILE = "chennai_100_new_checkpoint.csv"

CITY = "Chennai"
TARGET_NEW_HOTELS = 100
MAX_REVIEW_PAGES = 3

BASE_URL = "https://serpapi.com/search.json"

# ============================================================
# CHENNAI AREAS
# ============================================================

CHENNAI_AREAS = [
    "Central Chennai",
    "T Nagar",
    "Anna Nagar",
    "Adyar",
    "Velachery",
    "Guindy",
    "Nungambakkam",
    "Egmore",
    "Mylapore",
    "Alwarpet",
    "Besant Nagar",
    "Royapettah",
    "Triplicane",
    "Kodambakkam",
    "Vadapalani",
    "Ashok Nagar",
    "Koyambedu",
    "Kilpauk",
    "Porur",
    "Koyambedu",
    "OMR",
    "Sholinganallur",
    "Perungudi",
    "Thoraipakkam",
    "Karapakkam",
    "Navalur",
    "Siruseri",
    "Tambaram",
    "Chromepet",
    "Pallavaram",
    "Medavakkam",
    "Pallikaranai",
    "Perumbakkam",
    "Porur",
    "Ramapuram",
    "Manapakkam",
    "Mogappair",
    "Ambattur",
    "Avadi",
    "Poonamallee",
    "Chennai Airport",
    "ECR",
    "Neelankarai",
    "Palavakkam",
    "Injambakkam",
    "Kottivakkam",
    "Madhya Kailash",
    "Mount Road",
    "Chennai Central",
    "Perambur",
    "Royapuram",
    "Washermenpet",
    "George Town",
    "Broadway",
    "Chetpet",
    "Purasawalkam",
    "Saidapet",
    "Thiruvanmiyur",
    "Kotturpuram"
]

# Remove accidental duplicate areas
CHENNAI_AREAS = list(dict.fromkeys(CHENNAI_AREAS))


# ============================================================
# LOAD EXISTING DATASET
# ============================================================

if os.path.exists(OUTPUT_FILE):

    df = pd.read_csv(OUTPUT_FILE)

    print("\n" + "=" * 70)
    print("EXISTING DATASET")
    print("=" * 70)

    print(f"Existing rows: {len(df)}")

else:

    df = pd.DataFrame()

    print("No existing dataset found. Starting fresh.")


# ============================================================
# GET EXISTING HOTEL IDS
# ============================================================

if not df.empty and "hotel_id" in df.columns:

    existing_hotel_ids = set(
        df["hotel_id"]
        .dropna()
        .astype(str)
    )

else:

    existing_hotel_ids = set()


print(
    f"Existing unique hotels: "
    f"{len(existing_hotel_ids)}"
)

print(
    f"Target new hotels: "
    f"{TARGET_NEW_HOTELS}"
)

print(
    f"Expected unique hotels after run: "
    f"{len(existing_hotel_ids) + TARGET_NEW_HOTELS}"
)


# ============================================================
# LOAD CHECKPOINT
# ============================================================

if os.path.exists(CHECKPOINT_FILE):

    checkpoint = pd.read_csv(CHECKPOINT_FILE)

    completed_hotels = set(
        checkpoint.loc[
            checkpoint["status"] == "completed",
            "hotel_id"
        ]
        .astype(str)
    )

    print(
        f"Previously completed in this run: "
        f"{len(completed_hotels)}"
    )

else:

    checkpoint = pd.DataFrame(
        columns=[
            "hotel_id",
            "hotel_name",
            "search_area",
            "status"
        ]
    )

    completed_hotels = set()


# ============================================================
# SAVE CHECKPOINT
# ============================================================

def save_checkpoint(
    hotel_id,
    hotel_name,
    search_area,
    status
):

    global checkpoint

    hotel_id = str(hotel_id)

    # Remove previous entry
    checkpoint = checkpoint[
        checkpoint["hotel_id"].astype(str) != hotel_id
    ]

    new_checkpoint = pd.DataFrame([
        {
            "hotel_id": hotel_id,
            "hotel_name": hotel_name,
            "search_area": search_area,
            "status": status
        }
    ])

    checkpoint = pd.concat(
        [
            checkpoint,
            new_checkpoint
        ],
        ignore_index=True
    )

    checkpoint.to_csv(
        CHECKPOINT_FILE,
        index=False
    )


# ============================================================
# SAVE DATASET
# ============================================================

def save_dataset():

    global df

    if "hotel_id" in df.columns:

        df["hotel_id"] = df["hotel_id"].astype(str)

    if "review_id" in df.columns:

        df = df.drop_duplicates(
            subset=["review_id"]
        )

    else:

        df = df.drop_duplicates()

    df.to_csv(
        OUTPUT_FILE,
        index=False
    )

    print(
        f"Dataset saved: "
        f"{len(df)} rows | "
        f"{df['hotel_id'].nunique()} unique hotels"
    )


# ============================================================
# SEARCH HOTELS USING SERPAPI
# ============================================================

def search_hotels(search_area):

    params = {
        "engine": "google_hotels",
        "q": f"Hotels in {search_area}, Chennai",
        "check_in_date": "2026-10-15",
        "check_out_date": "2026-10-16",
        "adults": "2",
        "currency": "INR",
        "gl": "in",
        "hl": "en",
        "api_key": API_KEY
    }

    response = requests.get(
        BASE_URL,
        params=params,
        timeout=60
    )

    response.raise_for_status()

    data = response.json()

    if "error" in data:

        print(
            f"SerpApi error: {data['error']}"
        )

        return []

    return data.get(
        "properties",
        []
    )


# ============================================================
# EXTRACT LOCATION
# ============================================================

def extract_location(hotel):

    gps = hotel.get("gps_coordinates")

    if gps:

        latitude = gps.get("latitude")
        longitude = gps.get("longitude")

        if latitude is not None and longitude is not None:

            return (
                f"Latitude: {latitude}, "
                f"Longitude: {longitude}"
            )

    return hotel.get(
        "address",
        ""
    )


# ============================================================
# EXTRACT AMENITIES
# ============================================================

def extract_amenities(hotel):

    amenities = []

    # Google Hotels commonly provides amenities
    for amenity in hotel.get(
        "amenities",
        []
    ):

        if isinstance(amenity, str):

            amenities.append(amenity)

        elif isinstance(amenity, dict):

            name = (
                amenity.get("name")
                or amenity.get("title")
            )

            if name:

                amenities.append(
                    str(name)
                )

    # Remove duplicates
    amenities = list(
        dict.fromkeys(amenities)
    )

    return " | ".join(
        amenities
    )


# ============================================================
# EXTRACT PRICE
# ============================================================

def extract_price(hotel):

    # Different Google Hotels responses
    # can expose price differently.

    price = hotel.get(
        "rate_per_night",
        {}
    )

    if isinstance(price, dict):

        value = (
            price.get("lowest")
            or price.get("extracted_lowest")
        )

        if value is not None:

            return value

    # Alternative field
    if hotel.get("price") is not None:

        return hotel.get("price")

    return None


# ============================================================
# CREATE HOTEL INFORMATION
# ============================================================

def make_hotel_row(
    hotel,
    search_area
):

    hotel_id = (
        hotel.get("property_token")
        or hotel.get("hotel_id")
    )

    hotel_name = hotel.get(
        "name",
        "Unknown Hotel"
    )

    gps = hotel.get(
        "gps_coordinates",
        {}
    )

    latitude = gps.get(
        "latitude"
    )

    longitude = gps.get(
        "longitude"
    )

    row = {

        "hotel_name":
            hotel_name,

        "hotel_id":
            str(hotel_id),

        "city":
            CITY,

        "location":
            extract_location(hotel),

        "latitude":
            latitude,

        "longitude":
            longitude,

        "price_per_night":
            extract_price(hotel),

        "hotel_category":
            hotel.get(
                "hotel_class"
            ),

        "customer_rating":
            hotel.get(
                "overall_rating"
            ),

        "review_count":
            hotel.get(
                "reviews"
            ),

        "amenities":
            extract_amenities(hotel),

        "review_text":
            "",

        "individual_review_rating":
            None,

        "review_date":
            "",

        "review_source":
            "Google Hotels via SerpApi",

        "search_area":
            search_area
    }

    return row


# ============================================================
# FETCH REVIEWS
# ============================================================

def fetch_reviews(
    hotel,
    search_area
):

    hotel_id = (
        hotel.get("property_token")
        or hotel.get("hotel_id")
    )

    hotel_name = hotel.get(
        "name",
        "Unknown Hotel"
    )

    if not hotel_id:

        return []

    all_reviews = []

    next_page_token = None

    for page in range(
        1,
        MAX_REVIEW_PAGES + 1
    ):

        params = {

            "engine":
                "google_hotels_reviews",

            "property_token":
                hotel_id,

            "hl":
                "en",

            "sort_by":
                "2",

            "api_key":
                API_KEY
        }

        # If SerpApi gives a next-page token,
        # use it for the next request.
        if next_page_token:

            params["next_page_token"] = (
                next_page_token
            )

        try:

            response = requests.get(
                BASE_URL,
                params=params,
                timeout=60
            )

            response.raise_for_status()

            data = response.json()

        except Exception as e:

            print(
                f"Review request failed: {e}"
            )

            break

        if "error" in data:

            print(
                f"Review API error: "
                f"{data['error']}"
            )

            break

        reviews = data.get(
            "reviews",
            []
        )

        if not reviews:

            break

        # ------------------------------------------
        # CONVERT REVIEWS TO OUR DATASET FORMAT
        # ------------------------------------------

        for review in reviews:

            reviewer = review.get(
                "user",
                {}
            )

            review_id = (
                review.get("review_id")
                or review.get("id")
            )

            # If SerpApi doesn't provide a review ID,
            # create a deterministic one.
            if not review_id:

                review_id = (
                    f"{hotel_id}_"
                    f"{len(all_reviews)}"
                )

            review_rating = (
                review.get("rating")
                or review.get(
                    "overall_rating"
                )
            )

            review_text = (
                review.get("snippet")
                or review.get("text")
                or review.get(
                    "review_text"
                )
                or ""
            )

            review_date = (
                review.get("date")
                or review.get(
                    "relative_date"
                )
                or ""
            )

            row = {

                "hotel_name":
                    hotel_name,

                "hotel_id":
                    str(hotel_id),

                "city":
                    CITY,

                "location":
                    extract_location(
                        hotel
                    ),

                "latitude":
                    hotel.get(
                        "gps_coordinates",
                        {}
                    ).get(
                        "latitude"
                    ),

                "longitude":
                    hotel.get(
                        "gps_coordinates",
                        {}
                    ).get(
                        "longitude"
                    ),

                "price_per_night":
                    extract_price(
                        hotel
                    ),

                "hotel_category":
                    hotel.get(
                        "hotel_class"
                    ),

                "customer_rating":
                    hotel.get(
                        "overall_rating"
                    ),

                "review_count":
                    hotel.get(
                        "reviews"
                    ),

                "amenities":
                    extract_amenities(
                        hotel
                    ),

                "review_text":
                    review_text,

                "individual_review_rating":
                    review_rating,

                "review_date":
                    review_date,

                "review_source":
                    "Google Hotels Reviews via SerpApi",

                "search_area":
                    search_area,

                "review_id":
                    review_id
            }

            all_reviews.append(
                row
            )

        # ------------------------------------------
        # NEXT PAGE
        # ------------------------------------------

        next_page_token = data.get(
            "serpapi_pagination",
            {}
        ).get(
            "next_page_token"
        )

        if not next_page_token:

            break

        time.sleep(1)

    return all_reviews


# ============================================================
# MAIN SCRAPING PROCESS
# ============================================================

new_hotel_ids = set()

new_hotels_collected = 0


print("\n" + "=" * 70)
print("STARTING CHENNAI NEW-HOTEL SCRAPING")
print("=" * 70)

print(
    f"Existing unique hotels: "
    f"{len(existing_hotel_ids)}"
)

print(
    f"Need NEW hotels: "
    f"{TARGET_NEW_HOTELS}"
)

print("=" * 70)


# ============================================================
# LOOP THROUGH CHENNAI AREAS
# ============================================================

for search_area in CHENNAI_AREAS:

    # Stop once 100 new hotels are collected
    if new_hotels_collected >= TARGET_NEW_HOTELS:

        break

    print("\n")
    print("=" * 70)

    print(
        f"SEARCH AREA: {search_area}"
    )

    print(
        f"Progress: "
        f"{new_hotels_collected}/"
        f"{TARGET_NEW_HOTELS} new hotels"
    )

    print("=" * 70)


    # ========================================================
    # SEARCH HOTELS
    # ========================================================

    try:

        hotels = search_hotels(
            search_area
        )

    except Exception as e:

        print(
            f"Hotel search failed for "
            f"{search_area}: {e}"
        )

        continue


    print(
        f"Hotels returned: "
        f"{len(hotels)}"
    )


    # ========================================================
    # PROCESS EACH HOTEL
    # ========================================================

    for hotel in hotels:

        # Stop at 100
        if new_hotels_collected >= TARGET_NEW_HOTELS:

            break


        # ----------------------------------------------------
        # HOTEL ID
        # ----------------------------------------------------

        hotel_id = (
            hotel.get("property_token")
            or hotel.get("hotel_id")
        )

        hotel_name = hotel.get(
            "name",
            "Unknown Hotel"
        )


        if not hotel_id:

            print(
                f"Skipping {hotel_name} "
                f"- no hotel ID"
            )

            continue


        hotel_id = str(
            hotel_id
        )


        # ----------------------------------------------------
        # SKIP EXISTING HOTELS
        # ----------------------------------------------------

        if hotel_id in existing_hotel_ids:

            print(
                f"SKIP EXISTING: "
                f"{hotel_name}"
            )

            continue


        # ----------------------------------------------------
        # SKIP ALREADY FOUND DURING THIS RUN
        # ----------------------------------------------------

        if hotel_id in new_hotel_ids:

            continue


        print("\n" + "-" * 60)

        print(
            f"NEW HOTEL "
            f"{new_hotels_collected + 1}/"
            f"{TARGET_NEW_HOTELS}"
        )

        print(
            f"Hotel: {hotel_name}"
        )

        print(
            f"Hotel ID: {hotel_id}"
        )

        print(
            f"Area: {search_area}"
        )

        print("-" * 60)


        # ====================================================
        # FETCH REVIEWS
        # ====================================================

        try:

            reviews = fetch_reviews(
                hotel,
                search_area
            )

        except Exception as e:

            print(
                f"Review collection failed: {e}"
            )

            reviews = []


        # ====================================================
        # IF REVIEWS EXIST
        # ====================================================

        if reviews:

            new_df = pd.DataFrame(
                reviews
            )

            df = pd.concat(
                [
                    df,
                    new_df
                ],
                ignore_index=True
            )

            # Remove duplicate reviews
            if "review_id" in df.columns:

                df = df.drop_duplicates(
                    subset=["review_id"]
                )

            print(
                f"Reviews collected: "
                f"{len(reviews)}"
            )

        else:

            # ------------------------------------------------
            # SAVE HOTEL EVEN IF NO REVIEWS
            # ------------------------------------------------
            #
            # This ensures that we really collect
            # 100 NEW UNIQUE HOTELS.

            hotel_row = make_hotel_row(
                hotel,
                search_area
            )

            df = pd.concat(
                [
                    df,
                    pd.DataFrame([
                        hotel_row
                    ])
                ],
                ignore_index=True
            )

            print(
                "No reviews found. "
                "Hotel information saved."
            )


        # ====================================================
        # SAVE IMMEDIATELY
        # ====================================================

        save_dataset()


        # ====================================================
        # UPDATE TRACKING
        # ====================================================

        new_hotel_ids.add(
            hotel_id
        )

        existing_hotel_ids.add(
            hotel_id
        )

        new_hotels_collected += 1


        save_checkpoint(
            hotel_id,
            hotel_name,
            search_area,
            "completed"
        )


        print(
            f"SUCCESS: {hotel_name}"
        )

        print(
            f"New hotel progress: "
            f"{new_hotels_collected}/"
            f"{TARGET_NEW_HOTELS}"
        )

        print(
            f"Total unique hotels currently: "
            f"{df['hotel_id'].nunique()}"
        )


        # Small delay
        time.sleep(2)


# ============================================================
# FINAL SAVE
# ============================================================

save_dataset()


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("CHENNAI SCRAPING COMPLETED")
print("=" * 70)

print(
    f"New hotels collected: "
    f"{new_hotels_collected}"
)

print(
    f"Total unique hotels: "
    f"{df['hotel_id'].nunique()}"
)

print(
    f"Total dataset rows: "
    f"{len(df)}"
)

print(
    f"Output file: "
    f"{OUTPUT_FILE}"
)

print(
    f"Checkpoint file: "
    f"{CHECKPOINT_FILE}"
)

print("=" * 70)


# ============================================================
# FINAL HOTEL COUNT CHECK
# ============================================================

print("\nFINAL HOTEL COUNT")

hotel_counts = (
    df.groupby("hotel_id")
      .size()
      .reset_index(
          name="review_rows"
      )
)

print(
    f"Unique hotels: "
    f"{len(hotel_counts)}"
)

print(
    f"Total review/data rows: "
    f"{len(df)}"
)

print(
    "\nDataset is ready."
)

Enter your SerpApi API key: 8d59aa17fa2898d550e74c1f16359c11330612a259078cc83a930061089f75b1

EXISTING DATASET
Existing rows: 976
Existing unique hotels: 55
Target new hotels: 100
Expected unique hotels after run: 155

STARTING CHENNAI NEW-HOTEL SCRAPING
Existing unique hotels: 55
Need NEW hotels: 100


SEARCH AREA: Central Chennai
Progress: 0/100 new hotels
Hotels returned: 20
SKIP EXISTING: The Hydel Park (Business Class Hotel) Chennai
SKIP EXISTING: Lemon Tree Hotel, Shimona Chennai
SKIP EXISTING: Four Points by Sheraton Chennai OMR
SKIP EXISTING: Ramada by Wyndham Chennai Egmore
SKIP EXISTING: Holiday Inn Express Chennai Omr Thoraipakkam, an IHG Hotel
SKIP EXISTING: Hotel Greenpark Chennai

------------------------------------------------------------
NEW HOTEL 1/100
Hotel: Leisure Stays - Premium Suites - Three-Bedroom Apartment
Hotel ID: ChkQgNb04cbP2_Z-Gg0vZy8xMW1reXJfcnA5EAI
Area: Central Chennai
------------------------------------------------------------
Reviews collected: 6
D

/tmp/ipykernel_1058/1236986249.py:967: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(



------------------------------------------------------------
NEW HOTEL 20/100
Hotel: Golden Fruits Business Suites - Standard Double Room
Hotel ID: ChoQ7d2QyoSKpubjARoNL2cvMTF6MTQ0NGtsMhAC
Area: T Nagar
------------------------------------------------------------
Reviews collected: 4
Dataset saved: 499 rows | 20 unique hotels
SUCCESS: Golden Fruits Business Suites - Standard Double Room
New hotel progress: 20/100
Total unique hotels currently: 20


SEARCH AREA: Anna Nagar
Progress: 20/100 new hotels
Hotels returned: 20

------------------------------------------------------------
NEW HOTEL 21/100
Hotel: Blu Iris (Hot Plate)
Hotel ID: ChkIttOO5b2Yh-FaGg0vZy8xMWNuNXFocTBxEAE
Area: Anna Nagar
------------------------------------------------------------
Reviews collected: 30
Dataset saved: 529 rows | 21 unique hotels
SUCCESS: Blu Iris (Hot Plate)
New hotel progress: 21/100
Total unique hotels currently: 21
SKIP EXISTING: JP Hotel Chennai
SKIP EXISTING: Metro Grand Hotel

-----------------

/tmp/ipykernel_1058/1236986249.py:967: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(



------------------------------------------------------------
NEW HOTEL 36/100
Hotel: family and couple friendly and pet friendly
Hotel ID: ChoQj5D0_f7Oxb2YARoNL2cvMTF5M216MWx3OBAC
Area: Anna Nagar
------------------------------------------------------------
Review API error: Google Hotels hasn't returned any results for this query.
No reviews found. Hotel information saved.
Dataset saved: 919 rows | 34 unique hotels
SUCCESS: family and couple friendly and pet friendly
New hotel progress: 36/100
Total unique hotels currently: 34


...

Dataset saved: 2342 rows | 83 unique hotels
SUCCESS: Treebo Pechis Castle
New hotel progress: 98/100
Total unique hotels currently: 83

------------------------------------------------------------
NEW HOTEL 99/100
Hotel: The Belstead Hotel
Hotel ID: ChkIyqe8hYKInMY8Gg0vZy8xMWR4bGdjc2dfEAE
Area: Nungambakkam
------------------------------------------------------------
Review request failed: 429 Client Error: Too Many Requests for url: https://serpapi.com/search.json?engine=google_hotels_reviews&property_token=ChkIyqe8hYKInMY8Gg0vZy8xMWR4bGdjc2dfEAE&hl=en&sort_by=2&api_key=8d59aa17fa2898d550e74c1f16359c11330612a259078cc83a930061089f75b1
No reviews found. Hotel information saved.
Dataset saved: 2342 rows | 83 unique hotels
SUCCESS: The Belstead Hotel
New hotel progress: 99/100
Total unique hotels currently: 83


/tmp/ipykernel_1058/1236986249.py:967: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(



------------------------------------------------------------
NEW HOTEL 100/100
Hotel: Sweet Memories- Perfect home stay - Superior Double Room
Hotel ID: ChkQwZjQqO6bmoYtGg0vZy8xMXRnZ3ByNDJmEAI
Area: Nungambakkam
------------------------------------------------------------
Review request failed: 429 Client Error: Too Many Requests for url: https://serpapi.com/search.json?engine=google_hotels_reviews&property_token=ChkQwZjQqO6bmoYtGg0vZy8xMXRnZ3ByNDJmEAI&hl=en&sort_by=2&api_key=8d59aa17fa2898d550e74c1f16359c11330612a259078cc83a930061089f75b1
No reviews found. Hotel information saved.
Dataset saved: 2342 rows | 83 unique hotels


/tmp/ipykernel_1058/1236986249.py:967: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(


SUCCESS: Sweet Memories- Perfect home stay - Superior Double Room
New hotel progress: 100/100
Total unique hotels currently: 83
Dataset saved: 2342 rows | 83 unique hotels


CHENNAI SCRAPING COMPLETED
New hotels collected: 100
Total unique hotels: 83
Total dataset rows: 2342
Output file: chennai_hotel_dataset.csv
Checkpoint file: chennai_100_new_checkpoint.csv

FINAL HOTEL COUNT
Unique hotels: 83
Total review/data rows: 2342

Dataset is ready.


In [ ]:
from google.colab import files

files.download("chennai_hotel_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import time
import requests
import pandas as pd

# ============================================================
# CONFIGURATION
# ============================================================

API_KEY = input("Enter your SerpApi API key: ").strip()

# IMPORTANT:
# Keep using the SAME dataset file
OUTPUT_FILE = "chennai_hotel_dataset.csv"

# NEW checkpoint for this batch
CHECKPOINT_FILE = "chennai_next_100_checkpoint.csv"

CITY = "Chennai"

# ============================================================
# IMPORTANT
# ============================================================
# Your current dataset has 83 unique hotels.
#
# We now want 100 MORE NEW hotels.
#
# Expected final:
#
# 83 existing + 100 new = 183 unique hotels
# ============================================================

TARGET_NEW_HOTELS = 100

# Maximum review pages per hotel
MAX_REVIEW_PAGES = 3

BASE_URL = "https://serpapi.com/search.json"


# ============================================================
# CHENNAI SEARCH AREAS
# ============================================================

CHENNAI_AREAS = [

    # Central Chennai
    "Central Chennai",
    "Chennai Central",
    "Mount Road",
    "George Town",
    "Broadway",
    "Royapuram",
    "Washermenpet",

    # South / Central
    "T Nagar",
    "Nungambakkam",
    "Egmore",
    "Mylapore",
    "Alwarpet",
    "Royapettah",
    "Triplicane",
    "Chetpet",
    "Kilpauk",
    "Purasawalkam",
    "Saidapet",

    # West Chennai
    "Anna Nagar",
    "Mogappair",
    "Ambattur",
    "Avadi",
    "Poonamallee",
    "Porur",
    "Ramapuram",
    "Manapakkam",
    "Vadapalani",
    "Kodambakkam",
    "Ashok Nagar",
    "Koyambedu",

    # South Chennai
    "Guindy",
    "Adyar",
    "Besant Nagar",
    "Thiruvanmiyur",
    "Kotturpuram",
    "Madhya Kailash",
    "Velachery",
    "Pallikaranai",
    "Medavakkam",
    "Perumbakkam",

    # OMR
    "OMR",
    "Perungudi",
    "Thoraipakkam",
    "Karapakkam",
    "Sholinganallur",
    "Navalur",
    "Siruseri",

    # ECR
    "ECR",
    "Neelankarai",
    "Palavakkam",
    "Kottivakkam",
    "Kottivakkam ECR",
    "Injambakkam",

    # Tambaram side
    "Tambaram",
    "Chromepet",
    "Pallavaram",
    "Pammal",

    # Airport
    "Chennai Airport",

    # Northern / outer Chennai
    "Perambur",
    "Tondiarpet",
    "Madhavaram",
    "Red Hills",

    # Other important areas
    "Shenoy Nagar",
    "Aminjikarai",
    "Choolaimedu",
    "Nandanam",
    "Teynampet",
    "Thousand Lights",
    "Little Mount"
]

# Remove duplicate areas
CHENNAI_AREAS = list(
    dict.fromkeys(CHENNAI_AREAS)
)


# ============================================================
# LOAD EXISTING DATASET
# ============================================================

if os.path.exists(OUTPUT_FILE):

    df = pd.read_csv(
        OUTPUT_FILE
    )

    print("\n" + "=" * 70)
    print("EXISTING CHENNAI DATASET")
    print("=" * 70)

    print(
        f"Existing records: {len(df)}"
    )

else:

    print(
        f"\nERROR: {OUTPUT_FILE} not found."
    )

    print(
        "Make sure your existing Chennai CSV is "
        "uploaded to Colab."
    )

    raise FileNotFoundError(
        OUTPUT_FILE
    )


# ============================================================
# CHECK HOTEL_ID COLUMN
# ============================================================

if "hotel_id" not in df.columns:

    raise ValueError(
        "The dataset does not contain a 'hotel_id' column."
    )


# ============================================================
# GET EXISTING UNIQUE HOTEL IDS
# ============================================================

df["hotel_id"] = (
    df["hotel_id"]
    .astype(str)
)

existing_hotel_ids = set(
    df["hotel_id"]
    .dropna()
    .astype(str)
)


# Remove possible "nan"
existing_hotel_ids.discard(
    "nan"
)


print(
    f"Existing unique hotels: "
    f"{len(existing_hotel_ids)}"
)

print(
    f"New hotels required: "
    f"{TARGET_NEW_HOTELS}"
)

print(
    f"Expected final unique hotels: "
    f"{len(existing_hotel_ids) + TARGET_NEW_HOTELS}"
)

print("=" * 70)


# ============================================================
# LOAD NEW CHECKPOINT
# ============================================================

if os.path.exists(CHECKPOINT_FILE):

    checkpoint = pd.read_csv(
        CHECKPOINT_FILE
    )

    if not checkpoint.empty:

        completed_hotels = set(
            checkpoint.loc[
                checkpoint["status"] == "completed",
                "hotel_id"
            ]
            .astype(str)
        )

    else:

        completed_hotels = set()

    print(
        f"Hotels already completed in this batch: "
        f"{len(completed_hotels)}"
    )

else:

    checkpoint = pd.DataFrame(
        columns=[
            "hotel_id",
            "hotel_name",
            "search_area",
            "status"
        ]
    )

    completed_hotels = set()

    print(
        "No previous checkpoint found. "
        "Starting this batch fresh."
    )


# ============================================================
# PREVENT OLD CHECKPOINT HOTELS FROM BEING ADDED AGAIN
# ============================================================

known_hotel_ids = (
    existing_hotel_ids |
    completed_hotels
)


# ============================================================
# SAVE CHECKPOINT
# ============================================================

def save_checkpoint(
    hotel_id,
    hotel_name,
    search_area,
    status
):

    global checkpoint

    hotel_id = str(
        hotel_id
    )

    # Remove previous record for this hotel
    checkpoint = checkpoint[
        checkpoint["hotel_id"].astype(str)
        != hotel_id
    ]

    new_row = pd.DataFrame([
        {
            "hotel_id":
                hotel_id,

            "hotel_name":
                hotel_name,

            "search_area":
                search_area,

            "status":
                status
        }
    ])

    checkpoint = pd.concat(
        [
            checkpoint,
            new_row
        ],
        ignore_index=True
    )

    checkpoint.to_csv(
        CHECKPOINT_FILE,
        index=False
    )


# ============================================================
# SAVE DATASET
# ============================================================

def save_dataset():

    global df

    # Ensure hotel IDs are strings
    if "hotel_id" in df.columns:

        df["hotel_id"] = (
            df["hotel_id"]
            .astype(str)
        )

    # Remove duplicate reviews
    if "review_id" in df.columns:

        df = df.drop_duplicates(
            subset=["review_id"]
        )

    else:

        df = df.drop_duplicates()

    # Save
    df.to_csv(
        OUTPUT_FILE,
        index=False
    )

    print(
        f"Dataset saved: "
        f"{len(df)} records | "
        f"{df['hotel_id'].nunique()} unique hotels"
    )


# ============================================================
# SEARCH HOTELS USING SERPAPI
# ============================================================

def search_hotels(search_area):

    params = {

        "engine":
            "google_hotels",

        "q":
            f"Hotels in {search_area}, Chennai",

        "check_in_date":
            "2026-10-15",

        "check_out_date":
            "2026-10-16",

        "adults":
            "2",

        "currency":
            "INR",

        "gl":
            "in",

        "hl":
            "en",

        "api_key":
            API_KEY
    }

    response = requests.get(
        BASE_URL,
        params=params,
        timeout=60
    )

    response.raise_for_status()

    data = response.json()

    if "error" in data:

        print(
            f"SerpApi error: "
            f"{data['error']}"
        )

        return []

    return data.get(
        "properties",
        []
    )


# ============================================================
# EXTRACT LOCATION
# ============================================================

def extract_location(hotel):

    gps = hotel.get(
        "gps_coordinates"
    )

    if gps:

        latitude = gps.get(
            "latitude"
        )

        longitude = gps.get(
            "longitude"
        )

        if (
            latitude is not None
            and
            longitude is not None
        ):

            return (
                f"Latitude: {latitude}, "
                f"Longitude: {longitude}"
            )

    return hotel.get(
        "address",
        ""
    )


# ============================================================
# EXTRACT AMENITIES
# ============================================================

def extract_amenities(hotel):

    amenities = []

    for amenity in hotel.get(
        "amenities",
        []
    ):

        if isinstance(
            amenity,
            str
        ):

            amenities.append(
                amenity
            )

        elif isinstance(
            amenity,
            dict
        ):

            name = (
                amenity.get("name")
                or
                amenity.get("title")
            )

            if name:

                amenities.append(
                    str(name)
                )

    # Remove duplicate amenities
    amenities = list(
        dict.fromkeys(
            amenities
        )
    )

    return " | ".join(
        amenities
    )


# ============================================================
# EXTRACT PRICE
# ============================================================

def extract_price(hotel):

    price = hotel.get(
        "rate_per_night",
        {}
    )

    if isinstance(
        price,
        dict
    ):

        value = (
            price.get("lowest")
            or
            price.get("extracted_lowest")
        )

        if value is not None:

            return value

    # Alternative field
    if hotel.get(
        "price"
    ) is not None:

        return hotel.get(
            "price"
        )

    return None


# ============================================================
# CREATE HOTEL ROW
# ============================================================

def make_hotel_row(
    hotel,
    search_area
):

    hotel_id = (
        hotel.get(
            "property_token"
        )
        or
        hotel.get(
            "hotel_id"
        )
    )

    hotel_name = hotel.get(
        "name",
        "Unknown Hotel"
    )

    gps = hotel.get(
        "gps_coordinates",
        {}
    )

    latitude = gps.get(
        "latitude"
    )

    longitude = gps.get(
        "longitude"
    )

    return {

        "hotel_name":
            hotel_name,

        "hotel_id":
            str(hotel_id),

        "city":
            CITY,

        "location":
            extract_location(
                hotel
            ),

        "latitude":
            latitude,

        "longitude":
            longitude,

        "price_per_night":
            extract_price(
                hotel
            ),

        "hotel_category":
            hotel.get(
                "hotel_class"
            ),

        "customer_rating":
            hotel.get(
                "overall_rating"
            ),

        "review_count":
            hotel.get(
                "reviews"
            ),

        "amenities":
            extract_amenities(
                hotel
            ),

        "review_text":
            "",

        "individual_review_rating":
            None,

        "review_date":
            "",

        "review_source":
            "Google Hotels via SerpApi",

        "search_area":
            search_area,

        "review_id":
            ""
    }


# ============================================================
# FETCH REVIEWS
# ============================================================

def fetch_reviews(
    hotel,
    search_area
):

    hotel_id = (
        hotel.get(
            "property_token"
        )
        or
        hotel.get(
            "hotel_id"
        )
    )

    hotel_name = hotel.get(
        "name",
        "Unknown Hotel"
    )

    if not hotel_id:

        return []

    hotel_id = str(
        hotel_id
    )

    all_reviews = []

    next_page_token = None


    # ========================================================
    # REVIEW PAGES
    # ========================================================

    for page in range(
        1,
        MAX_REVIEW_PAGES + 1
    ):

        params = {

            "engine":
                "google_hotels_reviews",

            "property_token":
                hotel_id,

            "hl":
                "en",

            "sort_by":
                "2",

            "api_key":
                API_KEY
        }


        # Add pagination token
        if next_page_token:

            params[
                "next_page_token"
            ] = next_page_token


        try:

            response = requests.get(
                BASE_URL,
                params=params,
                timeout=60
            )

            response.raise_for_status()

            data = response.json()

        except Exception as e:

            print(
                f"Review request failed: {e}"
            )

            break


        # SerpApi error
        if "error" in data:

            print(
                f"Review API error: "
                f"{data['error']}"
            )

            break


        reviews = data.get(
            "reviews",
            []
        )


        # No reviews
        if not reviews:

            break


        # ====================================================
        # PROCESS REVIEWS
        # ====================================================

        for index, review in enumerate(
            reviews
        ):

            # -----------------------------------------------
            # REVIEW ID
            # -----------------------------------------------

            review_id = (
                review.get(
                    "review_id"
                )
                or
                review.get(
                    "id"
                )
            )

            # Create fallback ID
            if not review_id:

                review_id = (
                    f"{hotel_id}_"
                    f"{page}_"
                    f"{index}"
                )


            # -----------------------------------------------
            # REVIEW RATING
            # -----------------------------------------------

            review_rating = (
                review.get(
                    "rating"
                )
                or
                review.get(
                    "overall_rating"
                )
            )


            # -----------------------------------------------
            # REVIEW TEXT
            # -----------------------------------------------

            review_text = (

                review.get(
                    "snippet"
                )

                or

                review.get(
                    "text"
                )

                or

                review.get(
                    "review_text"
                )

                or
                ""
            )


            # -----------------------------------------------
            # REVIEW DATE
            # -----------------------------------------------

            review_date = (

                review.get(
                    "date"
                )

                or

                review.get(
                    "relative_date"
                )

                or
                ""
            )


            # -----------------------------------------------
            # CREATE ROW
            # -----------------------------------------------

            row = {

                "hotel_name":
                    hotel_name,

                "hotel_id":
                    hotel_id,

                "city":
                    CITY,

                "location":
                    extract_location(
                        hotel
                    ),

                "latitude":
                    hotel.get(
                        "gps_coordinates",
                        {}
                    ).get(
                        "latitude"
                    ),

                "longitude":
                    hotel.get(
                        "gps_coordinates",
                        {}
                    ).get(
                        "longitude"
                    ),

                "price_per_night":
                    extract_price(
                        hotel
                    ),

                "hotel_category":
                    hotel.get(
                        "hotel_class"
                    ),

                "customer_rating":
                    hotel.get(
                        "overall_rating"
                    ),

                "review_count":
                    hotel.get(
                        "reviews"
                    ),

                "amenities":
                    extract_amenities(
                        hotel
                    ),

                "review_text":
                    review_text,

                "individual_review_rating":
                    review_rating,

                "review_date":
                    review_date,

                "review_source":
                    "Google Hotels Reviews via SerpApi",

                "search_area":
                    search_area,

                "review_id":
                    review_id
            }

            all_reviews.append(
                row
            )


        # ====================================================
        # NEXT REVIEW PAGE
        # ====================================================

        next_page_token = (
            data.get(
                "serpapi_pagination",
                {}
            )
            .get(
                "next_page_token"
            )
        )


        if not next_page_token:

            break


        time.sleep(1)


    return all_reviews


# ============================================================
# MAIN SCRAPING PROCESS
# ============================================================

new_hotel_ids = set()

new_hotels_collected = 0


print("\n")
print("=" * 70)
print("STARTING CHENNAI - NEXT 100 NEW HOTELS")
print("=" * 70)

print(
    f"Existing unique hotels: "
    f"{len(existing_hotel_ids)}"
)

print(
    f"New hotels required: "
    f"{TARGET_NEW_HOTELS}"
)

print(
    f"Target after completion: "
    f"{len(existing_hotel_ids) + TARGET_NEW_HOTELS}"
)

print("=" * 70)


# ============================================================
# LOOP THROUGH AREAS
# ============================================================

for search_area in CHENNAI_AREAS:


    # Stop after 100 NEW hotels
    if (
        new_hotels_collected
        >=
        TARGET_NEW_HOTELS
    ):

        break


    print("\n")
    print("=" * 70)

    print(
        f"SEARCHING: {search_area}"
    )

    print(
        f"NEW HOTEL PROGRESS: "
        f"{new_hotels_collected}/"
        f"{TARGET_NEW_HOTELS}"
    )

    print("=" * 70)


    # ========================================================
    # SEARCH HOTELS
    # ========================================================

    try:

        hotels = search_hotels(
            search_area
        )

    except Exception as e:

        print(
            f"Hotel search failed: {e}"
        )

        continue


    print(
        f"Hotels returned: "
        f"{len(hotels)}"
    )


    # ========================================================
    # PROCESS HOTELS
    # ========================================================

    for hotel in hotels:


        # Stop at 100
        if (
            new_hotels_collected
            >=
            TARGET_NEW_HOTELS
        ):

            break


        # ====================================================
        # GET HOTEL ID
        # ====================================================

        hotel_id = (
            hotel.get(
                "property_token"
            )
            or
            hotel.get(
                "hotel_id"
            )
        )

        hotel_name = hotel.get(
            "name",
            "Unknown Hotel"
        )


        # No ID
        if not hotel_id:

            print(
                f"SKIP: {hotel_name} "
                f"- no hotel ID"
            )

            continue


        hotel_id = str(
            hotel_id
        )


        # ====================================================
        # SKIP EXISTING HOTEL
        # ====================================================

        if hotel_id in known_hotel_ids:

            print(
                f"SKIP EXISTING: "
                f"{hotel_name}"
            )

            continue


        # ====================================================
        # SKIP DUPLICATE IN THIS RUN
        # ====================================================

        if hotel_id in new_hotel_ids:

            continue


        # ====================================================
        # NEW HOTEL
        # ====================================================

        print("\n" + "-" * 60)

        print(
            f"NEW HOTEL "
            f"{new_hotels_collected + 1}/"
            f"{TARGET_NEW_HOTELS}"
        )

        print(
            f"Hotel: {hotel_name}"
        )

        print(
            f"Hotel ID: {hotel_id}"
        )

        print(
            f"Area: {search_area}"
        )

        print("-" * 60)


        # ====================================================
        # FETCH REVIEWS
        # ====================================================

        try:

            reviews = fetch_reviews(
                hotel,
                search_area
            )

        except Exception as e:

            print(
                f"Review collection failed: "
                f"{e}"
            )

            reviews = []


        # ====================================================
        # ADD REVIEWS
        # ====================================================

        if reviews:

            new_df = pd.DataFrame(
                reviews
            )

            df = pd.concat(
                [
                    df,
                    new_df
                ],
                ignore_index=True
            )

            print(
                f"Reviews collected: "
                f"{len(reviews)}"
            )


        # ====================================================
        # NO REVIEWS
        # ====================================================

        else:

            hotel_row = make_hotel_row(
                hotel,
                search_area
            )

            df = pd.concat(
                [
                    df,
                    pd.DataFrame([
                        hotel_row
                    ])
                ],
                ignore_index=True
            )

            print(
                "No reviews found."
            )

            print(
                "Hotel information saved."
            )


        # ====================================================
        # REMOVE DUPLICATE REVIEWS
        # ====================================================

        if "review_id" in df.columns:

            # Don't remove blank review IDs
            non_blank_review_ids = (
                df["review_id"]
                .fillna("")
                .astype(str)
                .str.strip()
                != ""
            )

            df_with_ids = df[
                non_blank_review_ids
            ].drop_duplicates(
                subset=["review_id"]
            )

            df_without_ids = df[
                ~non_blank_review_ids
            ]

            df = pd.concat(
                [
                    df_with_ids,
                    df_without_ids
                ],
                ignore_index=True
            )


        # ====================================================
        # SAVE IMMEDIATELY
        # ====================================================

        save_dataset()


        # ====================================================
        # UPDATE HOTEL TRACKING
        # ====================================================

        new_hotel_ids.add(
            hotel_id
        )

        existing_hotel_ids.add(
            hotel_id
        )

        known_hotel_ids.add(
            hotel_id
        )

        new_hotels_collected += 1


        # ====================================================
        # SAVE CHECKPOINT
        # ====================================================

        save_checkpoint(
            hotel_id,
            hotel_name,
            search_area,
            "completed"
        )


        # ====================================================
        # PROGRESS
        # ====================================================

        print(
            f"SUCCESS: {hotel_name}"
        )

        print(
            f"New hotels collected: "
            f"{new_hotels_collected}/"
            f"{TARGET_NEW_HOTELS}"
        )

        print(
            f"Total unique hotels now: "
            f"{df['hotel_id'].nunique()}"
        )

        print(
            f"Total records now: "
            f"{len(df)}"
        )


        # ====================================================
        # DELAY
        # ====================================================

        time.sleep(2)


# ============================================================
# FINAL SAVE
# ============================================================

save_dataset()


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("CHENNAI SCRAPING COMPLETED")
print("=" * 70)

print(
    f"New hotels collected: "
    f"{new_hotels_collected}"
)

print(
    f"Previous unique hotels: "
    f"{len(existing_hotel_ids) - new_hotels_collected}"
)

print(
    f"Total unique hotels: "
    f"{df['hotel_id'].nunique()}"
)

print(
    f"Total dataset rows: "
    f"{len(df)}"
)

print(
    f"Output file: "
    f"{OUTPUT_FILE}"
)

print(
    f"Checkpoint file: "
    f"{CHECKPOINT_FILE}"
)

print("=" * 70)


# ============================================================
# FINAL HOTEL COUNT
# ============================================================

print("\nFINAL HOTEL COUNT")

hotel_counts = (
    df.groupby(
        "hotel_id"
    )
    .size()
    .reset_index(
        name="review_rows"
    )
)


print(
    f"Unique hotels: "
    f"{len(hotel_counts)}"
)

print(
    f"Total review/data rows: "
    f"{len(df)}"
)


# ============================================================
# FINAL REVIEW STATISTICS
# ============================================================

if "review_text" in df.columns:

    review_text_count = (
        df["review_text"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
        .sum()
    )

    print(
        f"Records with review text: "
        f"{review_text_count}"
    )


if "individual_review_rating" in df.columns:

    review_rating_count = (
        df[
            "individual_review_rating"
        ]
        .notna()
        .sum()
    )

    print(
        f"Records with individual review rating: "
        f"{review_rating_count}"
    )


if "price_per_night" in df.columns:

    price_count = (
        df[
            "price_per_night"
        ]
        .notna()
        .sum()
    )

    print(
        f"Records with price: "
        f"{price_count}"
    )


print("\n" + "=" * 70)
print("DATASET IS READY")
print("=" * 70)

Enter your SerpApi API key: 419c18d17a96824ec591c0b5e0c2b73a83c0f66a38e0f68f53e7ecb61997fa58

EXISTING CHENNAI DATASET
Existing records: 2342
Existing unique hotels: 83
New hotels required: 100
Expected final unique hotels: 183
No previous checkpoint found. Starting this batch fresh.


STARTING CHENNAI - NEXT 100 NEW HOTELS
Existing unique hotels: 83
New hotels required: 100
Target after completion: 183


SEARCHING: Central Chennai
NEW HOTEL PROGRESS: 0/100
Hotels returned: 20

------------------------------------------------------------
NEW HOTEL 1/100
Hotel: Four Points by Sheraton Chennai OMR
Hotel ID: ChkIi-yZ2cj-tp8EGg0vZy8xMWZodHpkcnAzEAE
Area: Central Chennai
------------------------------------------------------------
Reviews collected: 30
Dataset saved: 2372 records | 84 unique hotels
SUCCESS: Four Points by Sheraton Chennai OMR
New hotels collected: 1/100
Total unique hotels now: 84
Total records now: 2372

------------------------------------------------------------
NEW HOTE

/tmp/ipykernel_1058/654340771.py:1242: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(



------------------------------------------------------------
NEW HOTEL 35/100
Hotel: The view signature studio
Hotel ID: ChoQ6o3MjL37ks24ARoNL2cvMTF4ZDJjc2dqdxAC
Area: Chennai Central
------------------------------------------------------------
Review API error: Google Hotels hasn't returned any results for this query.
No reviews found.
Hotel information saved.
Dataset saved: 3285 records | 117 unique hotels
SUCCESS: The view signature studio
New hotels collected: 35/100
Total unique hotels now: 117
Total records now: 3285


/tmp/ipykernel_1058/654340771.py:1242: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(




SEARCHING: Mount Road
NEW HOTEL PROGRESS: 35/100
Hotels returned: 20
SKIP EXISTING: Hablis - A Business Hotel In Chennal

------------------------------------------------------------
NEW HOTEL 36/100
Hotel: E Hotel Chennai
Hotel ID: ChkIvdfSt_qtxtqLARoML2cvMTJoc2Rra183EAE
Area: Mount Road
------------------------------------------------------------
Reviews collected: 30
Dataset saved: 3315 records | 118 unique hotels
SUCCESS: E Hotel Chennai
New hotels collected: 36/100
Total unique hotels now: 118
Total records now: 3315
SKIP EXISTING: Century Courtyard
SKIP EXISTING: GREEN TREE HOTEL

------------------------------------------------------------
NEW HOTEL 37/100
Hotel: RR MOUNT ELITE SUITES
Hotel ID: ChoI9KCayqvy0tmVARoNL2cvMTFzYnhkMzl0YhAB
Area: Mount Road
------------------------------------------------------------
Reviews collected: 30
Dataset saved: 3345 records | 119 unique hotels
SUCCESS: RR MOUNT ELITE SUITES
New hotels collected: 37/100
Total unique hotels now: 119
Total rec

...

Dataset saved: 4674 records | 166 unique hotels
SUCCESS: The Belstead Hotel
New hotels collected: 88/100
Total unique hotels now: 166
Total records now: 4674

------------------------------------------------------------
NEW HOTEL 89/100
Hotel: Sweet Memories- Perfect home stay - Superior Double Room
Hotel ID: ChkQwZjQqO6bmoYtGg0vZy8xMXRnZ3ByNDJmEAI
Area: Nungambakkam
------------------------------------------------------------
Review request failed: 429 Client Error: Too Many Requests for url: https://serpapi.com/search.json?engine=google_hotels_reviews&property_token=ChkQwZjQqO6bmoYtGg0vZy8xMXRnZ3ByNDJmEAI&hl=en&sort_by=2&api_key=419c18d17a96824ec591c0b5e0c2b73a83c0f66a38e0f68f53e7ecb61997fa58
No reviews found.
Hotel information saved.


/tmp/ipykernel_1058/654340771.py:1242: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(


Dataset saved: 4674 records | 166 unique hotels
SUCCESS: Sweet Memories- Perfect home stay - Superior Double Room
New hotels collected: 89/100
Total unique hotels now: 166
Total records now: 4674

------------------------------------------------------------
NEW HOTEL 90/100
Hotel: Redstone Service Apartment-TNagar Near UsConsulate
Hotel ID: ChkQuLGsoPXditd_Gg0vZy8xMXpoNHg2cWQ2EAI
Area: Nungambakkam
------------------------------------------------------------
Review request failed: 429 Client Error: Too Many Requests for url: https://serpapi.com/search.json?engine=google_hotels_reviews&property_token=ChkQuLGsoPXditd_Gg0vZy8xMXpoNHg2cWQ2EAI&hl=en&sort_by=2&api_key=419c18d17a96824ec591c0b5e0c2b73a83c0f66a38e0f68f53e7ecb61997fa58
No reviews found.
Hotel information saved.


/tmp/ipykernel_1058/654340771.py:1242: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(


Dataset saved: 4674 records | 166 unique hotels
SUCCESS: Redstone Service Apartment-TNagar Near UsConsulate
New hotels collected: 90/100
Total unique hotels now: 166
Total records now: 4674


SEARCHING: Egmore
NEW HOTEL PROGRESS: 90/100
Hotel search failed: 429 Client Error: Too Many Requests for url: https://serpapi.com/search.json?engine=google_hotels&q=Hotels+in+Egmore%2C+Chennai&check_in_date=2026-10-15&check_out_date=2026-10-16&adults=2&currency=INR&gl=in&hl=en&api_key=419c18d17a96824ec591c0b5e0c2b73a83c0f66a38e0f68f53e7ecb61997fa58


SEARCHING: Mylapore
NEW HOTEL PROGRESS: 90/100
Hotel search failed: 429 Client Error: Too Many Requests for url: https://serpapi.com/search.json?engine=google_hotels&q=Hotels+in+Mylapore%2C+Chennai&check_in_date=2026-10-15&check_out_date=2026-10-16&adults=2&currency=INR&gl=in&hl=en&api_key=419c18d17a96824ec591c0b5e0c2b73a83c0f66a38e0f68f53e7ecb61997fa58


SEARCHING: Alwarpet
NEW HOTEL PROGRESS: 90/100
Hotel search failed: 429 Client Error: Too Many R

In [ ]:
from google.colab import files

files.download("chennai_hotel_dataset.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import time
import requests
import pandas as pd

# ============================================================
# CONFIGURATION
# ============================================================

API_KEY = input("Enter your SerpApi API key: ").strip()

# IMPORTANT:
# Rename your current file to this exact name in Colab:
# andhra_pradesh_hotel_dataset.csv

OUTPUT_FILE = "andhra_pradesh_hotel_dataset.csv"

# NEW checkpoint for this batch
CHECKPOINT_FILE = "andhra_pradesh_next_100_checkpoint.csv"

STATE = "Andhra Pradesh"

# Collect 100 NEW hotels
TARGET_NEW_HOTELS = 100

# Maximum review pages per hotel
MAX_REVIEW_PAGES = 3

BASE_URL = "https://serpapi.com/search.json"


# ============================================================
# ANDHRA PRADESH SEARCH AREAS
# ============================================================

AP_AREAS = [

    # --------------------------------------------------------
    # VISAKHAPATNAM
    # --------------------------------------------------------

    "Visakhapatnam",
    "Vizag",
    "Dwaraka Nagar Visakhapatnam",
    "MVP Colony Visakhapatnam",
    "Siripuram Visakhapatnam",
    "Madhurawada Visakhapatnam",
    "Gajuwaka Visakhapatnam",
    "Rushikonda Visakhapatnam",
    "Beach Road Visakhapatnam",
    "RK Beach Visakhapatnam",
    "Waltair Visakhapatnam",
    "Visakhapatnam Airport",
    "Seethammadhara Visakhapatnam",
    "Akkayyapalem Visakhapatnam",
    "Dabagardens Visakhapatnam",
    "Kurmannapalem Visakhapatnam",

    # --------------------------------------------------------
    # VIJAYAWADA
    # --------------------------------------------------------

    "Vijayawada",
    "Benz Circle Vijayawada",
    "Governorpet Vijayawada",
    "Moghalrajpuram Vijayawada",
    "Labbipet Vijayawada",
    "Bhavanipuram Vijayawada",
    "Poranki Vijayawada",
    "Gollapudi Vijayawada",
    "Patamata Vijayawada",
    "Auto Nagar Vijayawada",

    # --------------------------------------------------------
    # TIRUPATI
    # --------------------------------------------------------

    "Tirupati",
    "Tirumala",
    "Renigunta",
    "Tiruchanur",
    "Kapila Theertham Tirupati",
    "Alipiri Tirupati",
    "Tirupati Railway Station",

    # --------------------------------------------------------
    # GUNTUR
    # --------------------------------------------------------

    "Guntur",
    "Brodipet Guntur",
    "Arundelpet Guntur",
    "Lakshmipuram Guntur",
    "Mangalagiri",
    "Amaravati",
    "Tadepalli",

    # --------------------------------------------------------
    # NELLORE
    # --------------------------------------------------------

    "Nellore",
    "Magunta Layout Nellore",
    "Dargamitta Nellore",
    "Vedayapalem Nellore",

    # --------------------------------------------------------
    # KAKINADA
    # --------------------------------------------------------

    "Kakinada",
    "Kakinada Beach",
    "Sarpavaram Kakinada",

    # --------------------------------------------------------
    # RAJAHMUNDRY
    # --------------------------------------------------------

    "Rajahmundry",
    "Rajamahendravaram",
    "Danavaipeta Rajahmundry",
    "Morampudi Rajahmundry",

    # --------------------------------------------------------
    # KURNOOL
    # --------------------------------------------------------

    "Kurnool",
    "Nandyal",
    "Kurnool Railway Station",

    # --------------------------------------------------------
    # ANANTAPUR
    # --------------------------------------------------------

    "Anantapur",
    "Anantapur Railway Station",

    # --------------------------------------------------------
    # KADAPA
    # --------------------------------------------------------

    "Kadapa",
    "Proddatur",
    "Rajampet",

    # --------------------------------------------------------
    # SRIKAKULAM
    # --------------------------------------------------------

    "Srikakulam",
    "Arasavalli",

    # --------------------------------------------------------
    # VIZIANAGARAM
    # --------------------------------------------------------

    "Vizianagaram",

    # --------------------------------------------------------
    # ELURU
    # --------------------------------------------------------

    "Eluru",

    # --------------------------------------------------------
    # MACHILIPATNAM
    # --------------------------------------------------------

    "Machilipatnam",

    # --------------------------------------------------------
    # BHIMAVARAM
    # --------------------------------------------------------

    "Bhimavaram",

    # --------------------------------------------------------
    # ONGOLE
    # --------------------------------------------------------

    "Ongole",

    # --------------------------------------------------------
    # BAPATLA / CHIRALA
    # --------------------------------------------------------

    "Bapatla",
    "Suryalanka",
    "Chirala",
    "Vodarevu",

    # --------------------------------------------------------
    # TOURIST / PILGRIMAGE LOCATIONS
    # --------------------------------------------------------

    "Kondapalli",
    "Lepakshi",
    "Srisailam",
    "Ahobilam",
    "Simhachalam",
    "Araku Valley",
    "Lambasingi",
    "Papikondalu",
    "Dwaraka Tirumala",

    # --------------------------------------------------------
    # OTHER AP LOCATIONS
    # --------------------------------------------------------

    "Narasaraopet",
    "Tenali",
    "Gudivada",
    "Nuzvid",
    "Tadepalligudem",
    "Tanuku",
    "Palakollu",
    "Amalapuram",
    "Mandapeta",
    "Samalkota",
    "Pithapuram",
    "Anakapalle",
    "Bobbili",
    "Parvathipuram",
    "Palasa",
    "Tekkali",
    "Markapur",
    "Chittoor",
    "Madanapalle",
    "Puttur",
    "Hindupur"
]

# Remove duplicate search areas
AP_AREAS = list(
    dict.fromkeys(AP_AREAS)
)


# ============================================================
# LOAD EXISTING DATASET
# ============================================================

if os.path.exists(OUTPUT_FILE):

    df = pd.read_csv(
        OUTPUT_FILE
    )

    print("\n" + "=" * 70)
    print("EXISTING ANDHRA PRADESH DATASET")
    print("=" * 70)

    print(
        f"Existing records: {len(df)}"
    )

else:

    print(
        f"\nERROR: {OUTPUT_FILE} not found."
    )

    print(
        "Upload/rename your existing Andhra Pradesh CSV "
        "to andhra_pradesh_hotel_dataset.csv"
    )

    raise FileNotFoundError(
        OUTPUT_FILE
    )


# ============================================================
# CHECK HOTEL_ID
# ============================================================

if "hotel_id" not in df.columns:

    raise ValueError(
        "The dataset does not contain a 'hotel_id' column."
    )


# ============================================================
# GET EXISTING UNIQUE HOTEL IDS
# ============================================================

df["hotel_id"] = (
    df["hotel_id"]
    .astype(str)
)

existing_hotel_ids = set(
    df["hotel_id"]
    .dropna()
    .astype(str)
)

# Remove invalid IDs
existing_hotel_ids.discard(
    "nan"
)

print(
    f"Existing unique hotels: "
    f"{len(existing_hotel_ids)}"
)

print(
    f"New hotels required: "
    f"{TARGET_NEW_HOTELS}"
)

print(
    f"Expected final unique hotels: "
    f"{len(existing_hotel_ids) + TARGET_NEW_HOTELS}"
)

print("=" * 70)


# ============================================================
# LOAD CHECKPOINT
# ============================================================

if os.path.exists(
    CHECKPOINT_FILE
):

    checkpoint = pd.read_csv(
        CHECKPOINT_FILE
    )

    if not checkpoint.empty:

        completed_hotels = set(
            checkpoint.loc[
                checkpoint["status"] == "completed",
                "hotel_id"
            ]
            .astype(str)
        )

    else:

        completed_hotels = set()

    print(
        f"Previously completed in this batch: "
        f"{len(completed_hotels)}"
    )

else:

    checkpoint = pd.DataFrame(
        columns=[
            "hotel_id",
            "hotel_name",
            "search_area",
            "status"
        ]
    )

    completed_hotels = set()

    print(
        "Starting a fresh batch."
    )


# ============================================================
# KNOWN HOTEL IDS
# ============================================================

known_hotel_ids = (
    existing_hotel_ids |
    completed_hotels
)


# ============================================================
# SAVE CHECKPOINT
# ============================================================

def save_checkpoint(
    hotel_id,
    hotel_name,
    search_area,
    status
):

    global checkpoint

    hotel_id = str(
        hotel_id
    )

    checkpoint = checkpoint[
        checkpoint["hotel_id"].astype(str)
        != hotel_id
    ]

    new_row = pd.DataFrame([
        {
            "hotel_id":
                hotel_id,

            "hotel_name":
                hotel_name,

            "search_area":
                search_area,

            "status":
                status
        }
    ])

    checkpoint = pd.concat(
        [
            checkpoint,
            new_row
        ],
        ignore_index=True
    )

    checkpoint.to_csv(
        CHECKPOINT_FILE,
        index=False
    )


# ============================================================
# SAVE DATASET
# ============================================================

def save_dataset():

    global df

    if "hotel_id" in df.columns:

        df["hotel_id"] = (
            df["hotel_id"]
            .astype(str)
        )

    # Remove duplicate reviews only when review_id exists
    if "review_id" in df.columns:

        valid_review_ids = (
            df["review_id"]
            .fillna("")
            .astype(str)
            .str.strip()
            != ""
        )

        df_with_ids = df[
            valid_review_ids
        ].drop_duplicates(
            subset=["review_id"]
        )

        df_without_ids = df[
            ~valid_review_ids
        ]

        df = pd.concat(
            [
                df_with_ids,
                df_without_ids
            ],
            ignore_index=True
        )

    else:

        df = df.drop_duplicates()

    df.to_csv(
        OUTPUT_FILE,
        index=False
    )

    print(
        f"Dataset saved: "
        f"{len(df)} records | "
        f"{df['hotel_id'].nunique()} unique hotels"
    )


# ============================================================
# SEARCH HOTELS
# ============================================================

def search_hotels(
    search_area
):

    params = {

        "engine":
            "google_hotels",

        "q":
            f"Hotels in {search_area}, Andhra Pradesh",

        "check_in_date":
            "2026-10-15",

        "check_out_date":
            "2026-10-16",

        "adults":
            "2",

        "currency":
            "INR",

        "gl":
            "in",

        "hl":
            "en",

        "api_key":
            API_KEY
    }

    response = requests.get(
        BASE_URL,
        params=params,
        timeout=60
    )

    response.raise_for_status()

    data = response.json()

    if "error" in data:

        print(
            f"SerpApi error: "
            f"{data['error']}"
        )

        return []

    return data.get(
        "properties",
        []
    )


# ============================================================
# EXTRACT LOCATION
# ============================================================

def extract_location(
    hotel
):

    gps = hotel.get(
        "gps_coordinates"
    )

    if gps:

        latitude = gps.get(
            "latitude"
        )

        longitude = gps.get(
            "longitude"
        )

        if (
            latitude is not None
            and
            longitude is not None
        ):

            return (
                f"Latitude: {latitude}, "
                f"Longitude: {longitude}"
            )

    return hotel.get(
        "address",
        ""
    )


# ============================================================
# EXTRACT AMENITIES
# ============================================================

def extract_amenities(
    hotel
):

    amenities = []

    for amenity in hotel.get(
        "amenities",
        []
    ):

        if isinstance(
            amenity,
            str
        ):

            amenities.append(
                amenity
            )

        elif isinstance(
            amenity,
            dict
        ):

            name = (
                amenity.get("name")
                or
                amenity.get("title")
            )

            if name:

                amenities.append(
                    str(name)
                )

    amenities = list(
        dict.fromkeys(
            amenities
        )
    )

    return " | ".join(
        amenities
    )


# ============================================================
# EXTRACT PRICE
# ============================================================

def extract_price(
    hotel
):

    price = hotel.get(
        "rate_per_night",
        {}
    )

    if isinstance(
        price,
        dict
    ):

        value = (
            price.get("lowest")
            or
            price.get("extracted_lowest")
        )

        if value is not None:

            return value

    if hotel.get(
        "price"
    ) is not None:

        return hotel.get(
            "price"
        )

    return None


# ============================================================
# CREATE HOTEL ROW
# ============================================================

def make_hotel_row(
    hotel,
    search_area
):

    hotel_id = (
        hotel.get(
            "property_token"
        )
        or
        hotel.get(
            "hotel_id"
        )
    )

    hotel_name = hotel.get(
        "name",
        "Unknown Hotel"
    )

    gps = hotel.get(
        "gps_coordinates",
        {}
    )

    latitude = gps.get(
        "latitude"
    )

    longitude = gps.get(
        "longitude"
    )

    return {

        "hotel_name":
            hotel_name,

        "hotel_id":
            str(hotel_id),

        "state":
            STATE,

        "location":
            extract_location(
                hotel
            ),

        "latitude":
            latitude,

        "longitude":
            longitude,

        "price_per_night":
            extract_price(
                hotel
            ),

        "hotel_category":
            hotel.get(
                "hotel_class"
            ),

        "customer_rating":
            hotel.get(
                "overall_rating"
            ),

        "review_count":
            hotel.get(
                "reviews"
            ),

        "amenities":
            extract_amenities(
                hotel
            ),

        "review_text":
            "",

        "individual_review_rating":
            None,

        "review_date":
            "",

        "review_source":
            "Google Hotels via SerpApi",

        "search_area":
            search_area,

        "review_id":
            ""
    }


# ============================================================
# FETCH REVIEWS
# ============================================================

def fetch_reviews(
    hotel,
    search_area
):

    hotel_id = (
        hotel.get(
            "property_token"
        )
        or
        hotel.get(
            "hotel_id"
        )
    )

    hotel_name = hotel.get(
        "name",
        "Unknown Hotel"
    )

    if not hotel_id:

        return []

    hotel_id = str(
        hotel_id
    )

    all_reviews = []

    next_page_token = None


    for page in range(
        1,
        MAX_REVIEW_PAGES + 1
    ):

        params = {

            "engine":
                "google_hotels_reviews",

            "property_token":
                hotel_id,

            "hl":
                "en",

            "sort_by":
                "2",

            "api_key":
                API_KEY
        }


        if next_page_token:

            params[
                "next_page_token"
            ] = next_page_token


        try:

            response = requests.get(
                BASE_URL,
                params=params,
                timeout=60
            )

            response.raise_for_status()

            data = response.json()

        except Exception as e:

            print(
                f"Review request failed: {e}"
            )

            break


        if "error" in data:

            print(
                f"Review API error: "
                f"{data['error']}"
            )

            break


        reviews = data.get(
            "reviews",
            []
        )


        if not reviews:

            break


        # ====================================================
        # PROCESS REVIEWS
        # ====================================================

        for index, review in enumerate(
            reviews
        ):

            review_id = (
                review.get(
                    "review_id"
                )
                or
                review.get(
                    "id"
                )
            )

            if not review_id:

                review_id = (
                    f"{hotel_id}_"
                    f"{page}_"
                    f"{index}"
                )


            review_rating = (
                review.get(
                    "rating"
                )
                or
                review.get(
                    "overall_rating"
                )
            )


            review_text = (

                review.get(
                    "snippet"
                )

                or

                review.get(
                    "text"
                )

                or

                review.get(
                    "review_text"
                )

                or
                ""
            )


            review_date = (

                review.get(
                    "date"
                )

                or

                review.get(
                    "relative_date"
                )

                or
                ""
            )


            row = {

                "hotel_name":
                    hotel_name,

                "hotel_id":
                    hotel_id,

                "state":
                    STATE,

                "location":
                    extract_location(
                        hotel
                    ),

                "latitude":
                    hotel.get(
                        "gps_coordinates",
                        {}
                    ).get(
                        "latitude"
                    ),

                "longitude":
                    hotel.get(
                        "gps_coordinates",
                        {}
                    ).get(
                        "longitude"
                    ),

                "price_per_night":
                    extract_price(
                        hotel
                    ),

                "hotel_category":
                    hotel.get(
                        "hotel_class"
                    ),

                "customer_rating":
                    hotel.get(
                        "overall_rating"
                    ),

                "review_count":
                    hotel.get(
                        "reviews"
                    ),

                "amenities":
                    extract_amenities(
                        hotel
                    ),

                "review_text":
                    review_text,

                "individual_review_rating":
                    review_rating,

                "review_date":
                    review_date,

                "review_source":
                    "Google Hotels Reviews via SerpApi",

                "search_area":
                    search_area,

                "review_id":
                    review_id
            }

            all_reviews.append(
                row
            )


        # ====================================================
        # NEXT PAGE
        # ====================================================

        next_page_token = (
            data.get(
                "serpapi_pagination",
                {}
            )
            .get(
                "next_page_token"
            )
        )


        if not next_page_token:

            break


        time.sleep(1)


    return all_reviews


# ============================================================
# MAIN SCRAPING PROCESS
# ============================================================

new_hotel_ids = set()

new_hotels_collected = 0


print("\n")
print("=" * 70)
print("ANDHRA PRADESH - NEXT 100 NEW HOTELS")
print("=" * 70)

print(
    f"Existing unique hotels: "
    f"{len(existing_hotel_ids)}"
)

print(
    f"New hotels required: "
    f"{TARGET_NEW_HOTELS}"
)

print(
    f"Final target: "
    f"{len(existing_hotel_ids) + TARGET_NEW_HOTELS}"
)

print("=" * 70)


# ============================================================
# LOOP THROUGH SEARCH AREAS
# ============================================================

for search_area in AP_AREAS:


    if (
        new_hotels_collected
        >=
        TARGET_NEW_HOTELS
    ):

        break


    print("\n")
    print("=" * 70)

    print(
        f"SEARCHING: {search_area}"
    )

    print(
        f"NEW HOTEL PROGRESS: "
        f"{new_hotels_collected}/"
        f"{TARGET_NEW_HOTELS}"
    )

    print("=" * 70)


    # ========================================================
    # SEARCH HOTELS
    # ========================================================

    try:

        hotels = search_hotels(
            search_area
        )

    except Exception as e:

        print(
            f"Hotel search failed: {e}"
        )

        continue


    print(
        f"Hotels returned: "
        f"{len(hotels)}"
    )


    # ========================================================
    # PROCESS EACH HOTEL
    # ========================================================

    for hotel in hotels:


        if (
            new_hotels_collected
            >=
            TARGET_NEW_HOTELS
        ):

            break


        # ====================================================
        # HOTEL ID
        # ====================================================

        hotel_id = (
            hotel.get(
                "property_token"
            )
            or
            hotel.get(
                "hotel_id"
            )
        )

        hotel_name = hotel.get(
            "name",
            "Unknown Hotel"
        )


        if not hotel_id:

            print(
                f"SKIP: {hotel_name} "
                f"- no hotel ID"
            )

            continue


        hotel_id = str(
            hotel_id
        )


        # ====================================================
        # SKIP EXISTING HOTEL
        # ====================================================

        if hotel_id in known_hotel_ids:

            print(
                f"SKIP EXISTING: "
                f"{hotel_name}"
            )

            continue


        # ====================================================
        # SKIP DUPLICATE FROM THIS RUN
        # ====================================================

        if hotel_id in new_hotel_ids:

            continue


        print("\n" + "-" * 60)

        print(
            f"NEW HOTEL "
            f"{new_hotels_collected + 1}/"
            f"{TARGET_NEW_HOTELS}"
        )

        print(
            f"Hotel: {hotel_name}"
        )

        print(
            f"Hotel ID: {hotel_id}"
        )

        print(
            f"Area: {search_area}"
        )

        print("-" * 60)


        # ====================================================
        # FETCH REVIEWS
        # ====================================================

        try:

            reviews = fetch_reviews(
                hotel,
                search_area
            )

        except Exception as e:

            print(
                f"Review collection failed: "
                f"{e}"
            )

            reviews = []


        # ====================================================
        # ADD REVIEW DATA
        # ====================================================

        if reviews:

            new_df = pd.DataFrame(
                reviews
            )

            df = pd.concat(
                [
                    df,
                    new_df
                ],
                ignore_index=True
            )

            print(
                f"Reviews collected: "
                f"{len(reviews)}"
            )


        # ====================================================
        # IF NO REVIEWS
        # ====================================================

        else:

            hotel_row = make_hotel_row(
                hotel,
                search_area
            )

            df = pd.concat(
                [
                    df,
                    pd.DataFrame([
                        hotel_row
                    ])
                ],
                ignore_index=True
            )

            print(
                "No reviews found."
            )

            print(
                "Hotel information saved."
            )


        # ====================================================
        # SAVE DATASET
        # ====================================================

        save_dataset()


        # ====================================================
        # UPDATE TRACKING
        # ====================================================

        new_hotel_ids.add(
            hotel_id
        )

        existing_hotel_ids.add(
            hotel_id
        )

        known_hotel_ids.add(
            hotel_id
        )

        new_hotels_collected += 1


        # ====================================================
        # SAVE CHECKPOINT
        # ====================================================

        save_checkpoint(
            hotel_id,
            hotel_name,
            search_area,
            "completed"
        )


        # ====================================================
        # PROGRESS
        # ====================================================

        print(
            f"SUCCESS: {hotel_name}"
        )

        print(
            f"New hotels: "
            f"{new_hotels_collected}/"
            f"{TARGET_NEW_HOTELS}"
        )

        print(
            f"Total unique hotels: "
            f"{df['hotel_id'].nunique()}"
        )

        print(
            f"Total records: "
            f"{len(df)}"
        )


        # ====================================================
        # DELAY
        # ====================================================

        time.sleep(2)


# ============================================================
# FINAL SAVE
# ============================================================

save_dataset()


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("ANDHRA PRADESH SCRAPING COMPLETED")
print("=" * 70)

print(
    f"New hotels collected: "
    f"{new_hotels_collected}"
)

print(
    f"Total unique hotels: "
    f"{df['hotel_id'].nunique()}"
)

print(
    f"Total dataset records: "
    f"{len(df)}"
)

print(
    f"Output file: "
    f"{OUTPUT_FILE}"
)

print(
    f"Checkpoint file: "
    f"{CHECKPOINT_FILE}"
)

print("=" * 70)


# ============================================================
# FINAL STATISTICS
# ============================================================

print("\nFINAL DATASET STATISTICS")

print(
    f"Unique hotels: "
    f"{df['hotel_id'].nunique()}"
)

print(
    f"Total records: "
    f"{len(df)}"
)


if "review_text" in df.columns:

    review_text_count = (
        df["review_text"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
        .sum()
    )

    print(
        f"Records with review text: "
        f"{review_text_count}"
    )


if "individual_review_rating" in df.columns:

    rating_count = (
        df[
            "individual_review_rating"
        ]
        .notna()
        .sum()
    )

    print(
        f"Records with individual review rating: "
        f"{rating_count}"
    )


if "price_per_night" in df.columns:

    price_count = (
        df[
            "price_per_night"
        ]
        .notna()
        .sum()
    )

    print(
        f"Records with price: "
        f"{price_count}"
    )


if "amenities" in df.columns:

    amenities_count = (
        df["amenities"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
        .sum()
    )

    print(
        f"Records with amenities: "
        f"{amenities_count}"
    )


print("\n")
print("=" * 70)
print("DATASET READY")
print("=" * 70)

Enter your SerpApi API key: c83390bd00bccbdebe7a00eee8b8d36bd529e9a9b553635a05f81dbee4d29763

EXISTING ANDHRA PRADESH DATASET
Existing records: 1126
Existing unique hotels: 72
New hotels required: 100
Expected final unique hotels: 172
Starting a fresh batch.


ANDHRA PRADESH - NEXT 100 NEW HOTELS
Existing unique hotels: 72
New hotels required: 100
Final target: 172


SEARCHING: Visakhapatnam
NEW HOTEL PROGRESS: 0/100
Hotels returned: 20
SKIP EXISTING: The Butterfly Luxury Serviced Apartments

------------------------------------------------------------
NEW HOTEL 1/100
Hotel: FabHotel Gita Residency
Hotel ID: ChkIiPjvnbTqrrE9Gg0vZy8xMW1zNm12Y2pyEAE
Area: Visakhapatnam
------------------------------------------------------------
Reviews collected: 22
Dataset saved: 1148 records | 73 unique hotels
SUCCESS: FabHotel Gita Residency
New hotels: 1/100
Total unique hotels: 73
Total records: 1148

------------------------------------------------------------
NEW HOTEL 2/100
Hotel: MAPLE COMFORT 

/tmp/ipykernel_1058/393633700.py:1328: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(



------------------------------------------------------------
NEW HOTEL 12/100
Hotel: AkSHARAINNOTEL
Hotel ID: ChoQ4cOm9IfStt-jARoNL2cvMTFudGhqYzRiORAC
Area: Visakhapatnam
------------------------------------------------------------
Review API error: Google Hotels hasn't returned any results for this query.
No reviews found.
Hotel information saved.
Dataset saved: 1407 records | 84 unique hotels
SUCCESS: AkSHARAINNOTEL
New hotels: 12/100
Total unique hotels: 84
Total records: 1407


SEARCHING: Vizag
NEW HOTEL PROGRESS: 12/100
Hotels returned: 20
SKIP EXISTING: Northway Hotel
SKIP EXISTING: The Port Hotel
SKIP EXISTING: Keys Select By Lemon Tree Hotel, Visakhapatnam
SKIP EXISTING: Hotel Rockdale
SKIP EXISTING: Treebo Raga
SKIP EXISTING: HOTEL TOUCHWOOD
SKIP EXISTING: Treebo Marina Grand

------------------------------------------------------------
NEW HOTEL 13/100
Hotel: Vibe Vacation Hotels
Hotel ID: ChkIt9DRz9vDvLUKGg0vZy8xMXczbW56em5xEAE
Area: Vizag
----------------------------------

/tmp/ipykernel_1058/393633700.py:1328: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(



------------------------------------------------------------
NEW HOTEL 32/100
Hotel: Servostay 5 Fully Furnished 2 BHK Flats in MVP Colony, Vizag
Hotel ID: ChoQ-v_urMjf8-G7ARoNL2cvMTF6eHRneWxqNBAC
Area: MVP Colony Visakhapatnam
------------------------------------------------------------
Review API error: Google Hotels hasn't returned any results for this query.
No reviews found.
Hotel information saved.
Dataset saved: 1891 records | 104 unique hotels
SUCCESS: Servostay 5 Fully Furnished 2 BHK Flats in MVP Colony, Vizag
New hotels: 32/100
Total unique hotels: 104
Total records: 1891


SEARCHING: Siripuram Visakhapatnam
NEW HOTEL PROGRESS: 32/100
Hotels returned: 20
SKIP EXISTING: Hotel Vivana
SKIP EXISTING: Waltair Abode
SKIP EXISTING: Hotel Rockdale
SKIP EXISTING: Zostel Visakhapatnam (Vizag)
SKIP EXISTING: Hotel Five Elements
SKIP EXISTING: Zora Park Hotel
SKIP EXISTING: HOTEL TOUCHWOOD
SKIP EXISTING: OYO Townhouse Visakhapatnam
SKIP EXISTING: Treebo Rainbow Grand
SKIP EXISTING: The

...


------------------------------------------------------------
NEW HOTEL 68/100
Hotel: Hotel Meena Paradise
Hotel ID: ChoI6ZK67t2imc_ZARoNL2cvMTFiYnJneGx0YhAB
Area: Gajuwaka Visakhapatnam
------------------------------------------------------------
Reviews collected: 30
Dataset saved: 2790 records | 140 unique hotels
SUCCESS: Hotel Meena Paradise
New hotels: 68/100
Total unique hotels: 140
Total records: 2790

------------------------------------------------------------
NEW HOTEL 69/100
Hotel: OYO Townhouse Gajuwaka Near CMR Mall
Hotel ID: ChkI153ApuOwzIMjGg0vZy8xMWI1eXRnOV9xEAE
Area: Gajuwaka Visakhapatnam
------------------------------------------------------------
Reviews collected: 30
Dataset saved: 2820 records | 141 unique hotels
SUCCESS: OYO Townhouse Gajuwaka Near CMR Mall
New hotels: 69/100
Total unique hotels: 141
Total records: 2820

------------------------------------------------------------
NEW HOTEL 70/100
Hotel: Hotel Phoenix Grand
Hotel ID: ChkI8bWf9Nja9c49Gg0vZy8xMW01Y

/tmp/ipykernel_1058/393633700.py:1328: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(


SKIP EXISTING: The Port Hotel
SKIP EXISTING: Hotel Rockdale

------------------------------------------------------------
NEW HOTEL 88/100
Hotel: Beach stay guest house
Hotel ID: ChoI8q_5hrW9yaiHARoNL2cvMTFwYms1anZyMRAB
Area: Beach Road Visakhapatnam
------------------------------------------------------------
Reviews collected: 30
Dataset saved: 3316 records | 160 unique hotels
SUCCESS: Beach stay guest house
New hotels: 88/100
Total unique hotels: 160
Total records: 3316
SKIP EXISTING: Hotel Emerald
SKIP EXISTING: Hotel Five Elements

------------------------------------------------------------
NEW HOTEL 89/100
Hotel: Sky Mountains Guest House
Hotel ID: ChoIruTl3qXp1eONARoNL2cvMTF4bmhoOXdwcxAB
Area: Beach Road Visakhapatnam
------------------------------------------------------------
Reviews collected: 30
Dataset saved: 3346 records | 161 unique hotels
SUCCESS: Sky Mountains Guest House
New hotels: 89/100
Total unique hotels: 161
Total records: 3346

---------------------------------

/tmp/ipykernel_1058/393633700.py:1328: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(


Dataset saved: 3359 records | 165 unique hotels
SUCCESS: 6 Beach facing Hotel Suites in Lovely R.K. Beach, Visakhapatnam, with AC, WiFi
New hotels: 93/100
Total unique hotels: 165
Total records: 3359
SKIP EXISTING: Northway Hotel
SKIP EXISTING: Treebo Premium The Regal Park
SKIP EXISTING: Hotel Kinnera Comforts
SKIP EXISTING: Gadiraju Palace Convention Center & Hotel

------------------------------------------------------------
NEW HOTEL 94/100
Hotel: Blue Moon Hotel
Hotel ID: ChkI-_aboMeZ-p9UGg0vZy8xMWd2MDc0YnAzEAE
Area: Beach Road Visakhapatnam
------------------------------------------------------------
Review request failed: 429 Client Error: Too Many Requests for url: https://serpapi.com/search.json?engine=google_hotels_reviews&property_token=ChkI-_aboMeZ-p9UGg0vZy8xMWd2MDc0YnAzEAE&hl=en&sort_by=2&api_key=c83390bd00bccbdebe7a00eee8b8d36bd529e9a9b553635a05f81dbee4d29763
No reviews found.
Hotel information saved.
Dataset saved: 3360 records | 166 unique hotels
SUCCESS: Blue Moon Hot

In [ ]:
from google.colab import files

files.download("andhra_pradesh_hotel_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import time
import requests
import pandas as pd

# ============================================================
# CONFIGURATION
# ============================================================

API_KEY = input("Enter your SerpApi API key: ").strip()

OUTPUT_FILE = "jaipur_hotel_dataset.csv"

# Separate checkpoint for Jaipur
CHECKPOINT_FILE = "jaipur_100_checkpoint.csv"

CITY = "Jaipur"
STATE = "Rajasthan"

# Number of UNIQUE Jaipur hotels required
TARGET_NEW_HOTELS = 100

# Maximum review pages per hotel
MAX_REVIEW_PAGES = 3

BASE_URL = "https://serpapi.com/search.json"


# ============================================================
# JAIPUR SEARCH AREAS
# ============================================================

JAIPUR_AREAS = [

    # --------------------------------------------------------
    # CENTRAL / BUSINESS AREAS
    # --------------------------------------------------------

    "C Scheme Jaipur",
    "MI Road Jaipur",
    "Ashok Nagar Jaipur",
    "Panch Batti Jaipur",
    "Statue Circle Jaipur",
    "Civil Lines Jaipur",
    "Bais Godam Jaipur",
    "22 Godam Jaipur",
    "Sodala Jaipur",
    "Hawa Sadak Jaipur",

    # --------------------------------------------------------
    # BANI PARK / RAILWAY STATION
    # --------------------------------------------------------

    "Bani Park Jaipur",
    "Jaipur Railway Station",
    "Gopalbari Jaipur",
    "Sindhi Camp Jaipur",
    "Station Road Jaipur",
    "Khatipura Jaipur",
    "Hasanpura Jaipur",
    "Shastri Nagar Jaipur",

    # --------------------------------------------------------
    # OLD CITY / HERITAGE AREAS
    # --------------------------------------------------------

    "Pink City Jaipur",
    "Old City Jaipur",
    "Hawa Mahal Jaipur",
    "City Palace Jaipur",
    "Johari Bazaar Jaipur",
    "Bapu Bazaar Jaipur",
    "Tripolia Bazaar Jaipur",
    "Chandpole Jaipur",
    "Ramganj Jaipur",
    "Choura Rasta Jaipur",

    # --------------------------------------------------------
    # RAJA PARK / ADARSH NAGAR
    # --------------------------------------------------------

    "Raja Park Jaipur",
    "Adarsh Nagar Jaipur",
    "Jawahar Nagar Jaipur",
    "Tilak Nagar Jaipur",
    "Moti Doongri Jaipur",

    # --------------------------------------------------------
    # MALVIYA NAGAR
    # --------------------------------------------------------

    "Malviya Nagar Jaipur",
    "JLN Marg Jaipur",
    "Jawahar Circle Jaipur",
    "Durgapura Jaipur",
    "Jagatpura Jaipur",
    "Sanganer Jaipur",

    # --------------------------------------------------------
    # AIRPORT AREA
    # --------------------------------------------------------

    "Jaipur Airport",
    "Jaipur International Airport",
    "Airport Road Jaipur",
    "Tonk Road Jaipur",
    "Pratap Nagar Jaipur",

    # --------------------------------------------------------
    # TONK ROAD
    # --------------------------------------------------------

    "Tonk Road Jaipur",
    "Gopalpura Jaipur",
    "Gopalpura Bypass Jaipur",
    "Triveni Nagar Jaipur",
    "Mahesh Nagar Jaipur",

    # --------------------------------------------------------
    # VAISHALI NAGAR / WEST JAIPUR
    # --------------------------------------------------------

    "Vaishali Nagar Jaipur",
    "Gandhi Path Jaipur",
    "Ajmer Road Jaipur",
    "Nirman Nagar Jaipur",
    "Shyam Nagar Jaipur",
    "Sodala Jaipur",
    "Dharam Nagar Jaipur",

    # --------------------------------------------------------
    # VIDHYADHAR NAGAR / NORTH JAIPUR
    # --------------------------------------------------------

    "Vidhyadhar Nagar Jaipur",
    "Murlipura Jaipur",
    "Jhotwara Jaipur",
    "Vishwakarma Industrial Area Jaipur",
    "Sikar Road Jaipur",

    # --------------------------------------------------------
    # MANSAROVAR
    # --------------------------------------------------------

    "Mansarovar Jaipur",
    "New Sanganer Road Jaipur",
    "Patrakar Colony Jaipur",
    "Madhyam Marg Jaipur",
    "Shipra Path Jaipur",

    # --------------------------------------------------------
    # AMER / HERITAGE BELT
    # --------------------------------------------------------

    "Amer Jaipur",
    "Amer Fort Jaipur",
    "Kukas Jaipur",
    "Delhi Road Jaipur",
    "Kunda Jaipur",
    "Jaigarh Fort Jaipur",
    "Nahargarh Jaipur",

    # --------------------------------------------------------
    # DELHI ROAD
    # --------------------------------------------------------

    "Delhi Road Jaipur",
    "Kukas",
    "Achrol Jaipur",
    "Chomu Jaipur",

    # --------------------------------------------------------
    # OUTER JAIPUR / TOURISM
    # --------------------------------------------------------

    "Kanak Vrindavan Jaipur",
    "Jaisinghpura Jaipur",
    "Sawai Jai Singh Highway Jaipur",
    "Agra Road Jaipur",
    "Ramgarh Road Jaipur",

    # --------------------------------------------------------
    # IMPORTANT LOCALITIES
    # --------------------------------------------------------

    "Crown Plaza Jaipur",
    "Rambagh Jaipur",
    "Bhawani Singh Road Jaipur",
    "Sawai Ram Singh Road Jaipur",
    "Prithviraj Road Jaipur",
    "Bhankrota Jaipur",
    "Mahapura Jaipur",
    "Mansarovar Extension Jaipur"
]

# Remove duplicate areas
JAIPUR_AREAS = list(
    dict.fromkeys(JAIPUR_AREAS)
)


# ============================================================
# LOAD EXISTING JAIPUR DATASET
# ============================================================

if os.path.exists(OUTPUT_FILE):

    df = pd.read_csv(
        OUTPUT_FILE
    )

    print("\n" + "=" * 70)
    print("EXISTING JAIPUR DATASET")
    print("=" * 70)

    print(
        f"Existing records: {len(df)}"
    )

else:

    df = pd.DataFrame()

    print(
        "No existing Jaipur dataset found."
    )

    print(
        "Starting a fresh Jaipur dataset."
    )


# ============================================================
# GET EXISTING HOTEL IDS
# ============================================================

if (
    not df.empty
    and
    "hotel_id" in df.columns
):

    df["hotel_id"] = (
        df["hotel_id"]
        .astype(str)
    )

    existing_hotel_ids = set(
        df["hotel_id"]
        .dropna()
        .astype(str)
    )

    existing_hotel_ids.discard(
        "nan"
    )

else:

    existing_hotel_ids = set()


print(
    f"Existing unique hotels: "
    f"{len(existing_hotel_ids)}"
)

print(
    f"New hotels required: "
    f"{TARGET_NEW_HOTELS}"
)

print(
    f"Expected final unique hotels: "
    f"{len(existing_hotel_ids) + TARGET_NEW_HOTELS}"
)

print("=" * 70)


# ============================================================
# LOAD CHECKPOINT
# ============================================================

if os.path.exists(
    CHECKPOINT_FILE
):

    checkpoint = pd.read_csv(
        CHECKPOINT_FILE
    )

    if not checkpoint.empty:

        completed_hotels = set(
            checkpoint.loc[
                checkpoint["status"] == "completed",
                "hotel_id"
            ]
            .astype(str)
        )

    else:

        completed_hotels = set()

    print(
        f"Previously completed in this batch: "
        f"{len(completed_hotels)}"
    )

else:

    checkpoint = pd.DataFrame(
        columns=[
            "hotel_id",
            "hotel_name",
            "search_area",
            "status"
        ]
    )

    completed_hotels = set()

    print(
        "No Jaipur checkpoint found."
    )

    print(
        "Starting fresh."
    )


# ============================================================
# KNOWN HOTEL IDS
# ============================================================

known_hotel_ids = (
    existing_hotel_ids |
    completed_hotels
)


# ============================================================
# SAVE CHECKPOINT
# ============================================================

def save_checkpoint(
    hotel_id,
    hotel_name,
    search_area,
    status
):

    global checkpoint

    hotel_id = str(
        hotel_id
    )

    # Remove previous entry
    checkpoint = checkpoint[
        checkpoint["hotel_id"].astype(str)
        != hotel_id
    ]

    new_row = pd.DataFrame([
        {
            "hotel_id":
                hotel_id,

            "hotel_name":
                hotel_name,

            "search_area":
                search_area,

            "status":
                status
        }
    ])

    checkpoint = pd.concat(
        [
            checkpoint,
            new_row
        ],
        ignore_index=True
    )

    checkpoint.to_csv(
        CHECKPOINT_FILE,
        index=False
    )


# ============================================================
# SAVE DATASET
# ============================================================

def save_dataset():

    global df

    # Ensure hotel ID is string
    if "hotel_id" in df.columns:

        df["hotel_id"] = (
            df["hotel_id"]
            .astype(str)
        )

    # Remove duplicate review IDs
    if "review_id" in df.columns:

        valid_review_ids = (
            df["review_id"]
            .fillna("")
            .astype(str)
            .str.strip()
            != ""
        )

        df_with_ids = df[
            valid_review_ids
        ].drop_duplicates(
            subset=["review_id"]
        )

        df_without_ids = df[
            ~valid_review_ids
        ]

        df = pd.concat(
            [
                df_with_ids,
                df_without_ids
            ],
            ignore_index=True
        )

    else:

        df = df.drop_duplicates()

    # Save
    df.to_csv(
        OUTPUT_FILE,
        index=False
    )

    print(
        f"Dataset saved: "
        f"{len(df)} records | "
        f"{df['hotel_id'].nunique()} unique hotels"
    )


# ============================================================
# SEARCH HOTELS USING SERPAPI
# ============================================================

def search_hotels(
    search_area
):

    params = {

        "engine":
            "google_hotels",

        "q":
            f"Hotels in {search_area}, Jaipur, Rajasthan",

        "check_in_date":
            "2026-10-15",

        "check_out_date":
            "2026-10-16",

        "adults":
            "2",

        "currency":
            "INR",

        "gl":
            "in",

        "hl":
            "en",

        "api_key":
            API_KEY
    }

    response = requests.get(
        BASE_URL,
        params=params,
        timeout=60
    )

    response.raise_for_status()

    data = response.json()

    if "error" in data:

        print(
            f"SerpApi error: "
            f"{data['error']}"
        )

        return []

    return data.get(
        "properties",
        []
    )


# ============================================================
# EXTRACT LOCATION
# ============================================================

def extract_location(
    hotel
):

    gps = hotel.get(
        "gps_coordinates"
    )

    if gps:

        latitude = gps.get(
            "latitude"
        )

        longitude = gps.get(
            "longitude"
        )

        if (
            latitude is not None
            and
            longitude is not None
        ):

            return (
                f"Latitude: {latitude}, "
                f"Longitude: {longitude}"
            )

    return hotel.get(
        "address",
        ""
    )


# ============================================================
# EXTRACT AMENITIES
# ============================================================

def extract_amenities(
    hotel
):

    amenities = []

    for amenity in hotel.get(
        "amenities",
        []
    ):

        if isinstance(
            amenity,
            str
        ):

            amenities.append(
                amenity
            )

        elif isinstance(
            amenity,
            dict
        ):

            name = (
                amenity.get("name")
                or
                amenity.get("title")
            )

            if name:

                amenities.append(
                    str(name)
                )

    # Remove duplicate amenities
    amenities = list(
        dict.fromkeys(
            amenities
        )
    )

    return " | ".join(
        amenities
    )


# ============================================================
# EXTRACT PRICE
# ============================================================

def extract_price(
    hotel
):

    price = hotel.get(
        "rate_per_night",
        {}
    )

    if isinstance(
        price,
        dict
    ):

        value = (
            price.get("lowest")
            or
            price.get("extracted_lowest")
        )

        if value is not None:

            return value

    if hotel.get(
        "price"
    ) is not None:

        return hotel.get(
            "price"
        )

    return None


# ============================================================
# CREATE HOTEL ROW
# ============================================================

def make_hotel_row(
    hotel,
    search_area
):

    hotel_id = (
        hotel.get(
            "property_token"
        )
        or
        hotel.get(
            "hotel_id"
        )
    )

    hotel_name = hotel.get(
        "name",
        "Unknown Hotel"
    )

    gps = hotel.get(
        "gps_coordinates",
        {}
    )

    latitude = gps.get(
        "latitude"
    )

    longitude = gps.get(
        "longitude"
    )

    return {

        "hotel_name":
            hotel_name,

        "hotel_id":
            str(hotel_id),

        "city":
            CITY,

        "state":
            STATE,

        "location":
            extract_location(
                hotel
            ),

        "latitude":
            latitude,

        "longitude":
            longitude,

        "price_per_night":
            extract_price(
                hotel
            ),

        "hotel_category":
            hotel.get(
                "hotel_class"
            ),

        "customer_rating":
            hotel.get(
                "overall_rating"
            ),

        "review_count":
            hotel.get(
                "reviews"
            ),

        "amenities":
            extract_amenities(
                hotel
            ),

        "review_text":
            "",

        "individual_review_rating":
            None,

        "review_date":
            "",

        "review_source":
            "Google Hotels via SerpApi",

        "search_area":
            search_area,

        "review_id":
            ""
    }


# ============================================================
# FETCH REVIEWS
# ============================================================

def fetch_reviews(
    hotel,
    search_area
):

    hotel_id = (
        hotel.get(
            "property_token"
        )
        or
        hotel.get(
            "hotel_id"
        )
    )

    hotel_name = hotel.get(
        "name",
        "Unknown Hotel"
    )

    if not hotel_id:

        return []

    hotel_id = str(
        hotel_id
    )

    all_reviews = []

    next_page_token = None


    # ========================================================
    # REVIEW PAGES
    # ========================================================

    for page in range(
        1,
        MAX_REVIEW_PAGES + 1
    ):

        params = {

            "engine":
                "google_hotels_reviews",

            "property_token":
                hotel_id,

            "hl":
                "en",

            "sort_by":
                "2",

            "api_key":
                API_KEY
        }


        if next_page_token:

            params[
                "next_page_token"
            ] = next_page_token


        try:

            response = requests.get(
                BASE_URL,
                params=params,
                timeout=60
            )

            response.raise_for_status()

            data = response.json()

        except Exception as e:

            print(
                f"Review request failed: "
                f"{e}"
            )

            break


        if "error" in data:

            print(
                f"Review API error: "
                f"{data['error']}"
            )

            break


        reviews = data.get(
            "reviews",
            []
        )


        if not reviews:

            break


        # ====================================================
        # PROCESS REVIEWS
        # ====================================================

        for index, review in enumerate(
            reviews
        ):

            # ------------------------------------------------
            # REVIEW ID
            # ------------------------------------------------

            review_id = (
                review.get(
                    "review_id"
                )
                or
                review.get(
                    "id"
                )
            )

            if not review_id:

                review_id = (
                    f"{hotel_id}_"
                    f"{page}_"
                    f"{index}"
                )


            # ------------------------------------------------
            # REVIEW RATING
            # ------------------------------------------------

            review_rating = (
                review.get(
                    "rating"
                )
                or
                review.get(
                    "overall_rating"
                )
            )


            # ------------------------------------------------
            # REVIEW TEXT
            # ------------------------------------------------

            review_text = (

                review.get(
                    "snippet"
                )

                or

                review.get(
                    "text"
                )

                or

                review.get(
                    "review_text"
                )

                or
                ""
            )


            # ------------------------------------------------
            # REVIEW DATE
            # ------------------------------------------------

            review_date = (

                review.get(
                    "date"
                )

                or

                review.get(
                    "relative_date"
                )

                or
                ""
            )


            # ------------------------------------------------
            # CREATE REVIEW ROW
            # ------------------------------------------------

            row = {

                "hotel_name":
                    hotel_name,

                "hotel_id":
                    hotel_id,

                "city":
                    CITY,

                "state":
                    STATE,

                "location":
                    extract_location(
                        hotel
                    ),

                "latitude":
                    hotel.get(
                        "gps_coordinates",
                        {}
                    ).get(
                        "latitude"
                    ),

                "longitude":
                    hotel.get(
                        "gps_coordinates",
                        {}
                    ).get(
                        "longitude"
                    ),

                "price_per_night":
                    extract_price(
                        hotel
                    ),

                "hotel_category":
                    hotel.get(
                        "hotel_class"
                    ),

                "customer_rating":
                    hotel.get(
                        "overall_rating"
                    ),

                "review_count":
                    hotel.get(
                        "reviews"
                    ),

                "amenities":
                    extract_amenities(
                        hotel
                    ),

                "review_text":
                    review_text,

                "individual_review_rating":
                    review_rating,

                "review_date":
                    review_date,

                "review_source":
                    "Google Hotels Reviews via SerpApi",

                "search_area":
                    search_area,

                "review_id":
                    review_id
            }

            all_reviews.append(
                row
            )


        # ====================================================
        # NEXT REVIEW PAGE
        # ====================================================

        next_page_token = (
            data.get(
                "serpapi_pagination",
                {}
            )
            .get(
                "next_page_token"
            )
        )


        if not next_page_token:

            break


        time.sleep(1)


    return all_reviews


# ============================================================
# MAIN SCRAPING PROCESS
# ============================================================

new_hotel_ids = set()

new_hotels_collected = 0


print("\n")
print("=" * 70)
print("STARTING JAIPUR HOTEL SCRAPING")
print("=" * 70)

print(
    f"Existing unique hotels: "
    f"{len(existing_hotel_ids)}"
)

print(
    f"New hotels required: "
    f"{TARGET_NEW_HOTELS}"
)

print(
    f"Target after completion: "
    f"{len(existing_hotel_ids) + TARGET_NEW_HOTELS}"
)

print("=" * 70)


# ============================================================
# LOOP THROUGH JAIPUR AREAS
# ============================================================

for search_area in JAIPUR_AREAS:


    # Stop at 100 NEW hotels
    if (
        new_hotels_collected
        >=
        TARGET_NEW_HOTELS
    ):

        break


    print("\n")
    print("=" * 70)

    print(
        f"SEARCHING: {search_area}"
    )

    print(
        f"NEW HOTEL PROGRESS: "
        f"{new_hotels_collected}/"
        f"{TARGET_NEW_HOTELS}"
    )

    print("=" * 70)


    # ========================================================
    # SEARCH HOTELS
    # ========================================================

    try:

        hotels = search_hotels(
            search_area
        )

    except Exception as e:

        print(
            f"Hotel search failed: "
            f"{e}"
        )

        continue


    print(
        f"Hotels returned: "
        f"{len(hotels)}"
    )


    # ========================================================
    # PROCESS EACH HOTEL
    # ========================================================

    for hotel in hotels:


        # Stop at 100
        if (
            new_hotels_collected
            >=
            TARGET_NEW_HOTELS
        ):

            break


        # ====================================================
        # HOTEL ID
        # ====================================================

        hotel_id = (
            hotel.get(
                "property_token"
            )
            or
            hotel.get(
                "hotel_id"
            )
        )

        hotel_name = hotel.get(
            "name",
            "Unknown Hotel"
        )


        if not hotel_id:

            print(
                f"SKIP: {hotel_name} "
                f"- no hotel ID"
            )

            continue


        hotel_id = str(
            hotel_id
        )


        # ====================================================
        # SKIP EXISTING HOTEL
        # ====================================================

        if hotel_id in known_hotel_ids:

            print(
                f"SKIP EXISTING: "
                f"{hotel_name}"
            )

            continue


        # ====================================================
        # SKIP DUPLICATE IN CURRENT RUN
        # ====================================================

        if hotel_id in new_hotel_ids:

            continue


        # ====================================================
        # NEW HOTEL
        # ====================================================

        print("\n" + "-" * 60)

        print(
            f"NEW HOTEL "
            f"{new_hotels_collected + 1}/"
            f"{TARGET_NEW_HOTELS}"
        )

        print(
            f"Hotel: {hotel_name}"
        )

        print(
            f"Hotel ID: {hotel_id}"
        )

        print(
            f"Area: {search_area}"
        )

        print("-" * 60)


        # ====================================================
        # FETCH REVIEWS
        # ====================================================

        try:

            reviews = fetch_reviews(
                hotel,
                search_area
            )

        except Exception as e:

            print(
                f"Review collection failed: "
                f"{e}"
            )

            reviews = []


        # ====================================================
        # ADD REVIEW DATA
        # ====================================================

        if reviews:

            new_df = pd.DataFrame(
                reviews
            )

            df = pd.concat(
                [
                    df,
                    new_df
                ],
                ignore_index=True
            )

            print(
                f"Reviews collected: "
                f"{len(reviews)}"
            )


        # ====================================================
        # NO REVIEWS
        # ====================================================

        else:

            hotel_row = make_hotel_row(
                hotel,
                search_area
            )

            df = pd.concat(
                [
                    df,
                    pd.DataFrame([
                        hotel_row
                    ])
                ],
                ignore_index=True
            )

            print(
                "No reviews found."
            )

            print(
                "Hotel information saved."
            )


        # ====================================================
        # SAVE DATASET IMMEDIATELY
        # ====================================================

        save_dataset()


        # ====================================================
        # UPDATE TRACKING
        # ====================================================

        new_hotel_ids.add(
            hotel_id
        )

        existing_hotel_ids.add(
            hotel_id
        )

        known_hotel_ids.add(
            hotel_id
        )

        new_hotels_collected += 1


        # ====================================================
        # SAVE CHECKPOINT
        # ====================================================

        save_checkpoint(
            hotel_id,
            hotel_name,
            search_area,
            "completed"
        )


        # ====================================================
        # SHOW PROGRESS
        # ====================================================

        print(
            f"SUCCESS: {hotel_name}"
        )

        print(
            f"New hotels collected: "
            f"{new_hotels_collected}/"
            f"{TARGET_NEW_HOTELS}"
        )

        print(
            f"Total unique hotels now: "
            f"{df['hotel_id'].nunique()}"
        )

        print(
            f"Total records now: "
            f"{len(df)}"
        )


        # ====================================================
        # DELAY
        # ====================================================

        time.sleep(2)


# ============================================================
# FINAL SAVE
# ============================================================

save_dataset()


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("JAIPUR SCRAPING COMPLETED")
print("=" * 70)

print(
    f"New hotels collected: "
    f"{new_hotels_collected}"
)

print(
    f"Total unique hotels: "
    f"{df['hotel_id'].nunique()}"
)

print(
    f"Total dataset records: "
    f"{len(df)}"
)

print(
    f"Output file: "
    f"{OUTPUT_FILE}"
)

print(
    f"Checkpoint file: "
    f"{CHECKPOINT_FILE}"
)

print("=" * 70)


# ============================================================
# FINAL HOTEL COUNT
# ============================================================

print("\nFINAL HOTEL COUNT")

hotel_counts = (
    df.groupby(
        "hotel_id"
    )
    .size()
    .reset_index(
        name="review_rows"
    )
)

print(
    f"Unique hotels: "
    f"{len(hotel_counts)}"
)

print(
    f"Total review/data rows: "
    f"{len(df)}"
)


# ============================================================
# REVIEW STATISTICS
# ============================================================

if "review_text" in df.columns:

    review_text_count = (
        df["review_text"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
        .sum()
    )

    print(
        f"Records with review text: "
        f"{review_text_count}"
    )


if "individual_review_rating" in df.columns:

    rating_count = (
        df[
            "individual_review_rating"
        ]
        .notna()
        .sum()
    )

    print(
        f"Records with individual review rating: "
        f"{rating_count}"
    )


if "price_per_night" in df.columns:

    price_count = (
        df[
            "price_per_night"
        ]
        .notna()
        .sum()
    )

    print(
        f"Records with price: "
        f"{price_count}"
    )


if "amenities" in df.columns:

    amenities_count = (
        df["amenities"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
        .sum()
    )

    print(
        f"Records with amenities: "
        f"{amenities_count}"
    )


print("\n")
print("=" * 70)
print("JAIPUR DATASET READY")
print("=" * 70)

Enter your SerpApi API key: 91e58b9815a6b81599b5a81053c867df1c631a144f8439d494ea7b46261a71c6
No existing Jaipur dataset found.
Starting a fresh Jaipur dataset.
Existing unique hotels: 0
New hotels required: 100
Expected final unique hotels: 100
No Jaipur checkpoint found.
Starting fresh.


STARTING JAIPUR HOTEL SCRAPING
Existing unique hotels: 0
New hotels required: 100
Target after completion: 100


SEARCHING: C Scheme Jaipur
NEW HOTEL PROGRESS: 0/100
Hotels returned: 20

------------------------------------------------------------
NEW HOTEL 1/100
Hotel: Blue Beds Jaipur
Hotel ID: ChoI-YWyjJWBvaS7ARoNL2cvMTFjNnp3enF0aBAB
Area: C Scheme Jaipur
------------------------------------------------------------
Reviews collected: 30
Dataset saved: 30 records | 1 unique hotels
SUCCESS: Blue Beds Jaipur
New hotels collected: 1/100
Total unique hotels now: 1
Total records now: 30

------------------------------------------------------------
NEW HOTEL 2/100
Hotel: Flute Boutique By Hotel 91
Hotel 

/tmp/ipykernel_1058/801334368.py:1332: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(



------------------------------------------------------------
NEW HOTEL 20/100
Hotel: Welcome to Pinterest Inspired Villa in center of the city.
Hotel ID: ChoQh4Wq8_WCho7pARoNL2cvMTF5MWMzZDl5MRAC
Area: C Scheme Jaipur
------------------------------------------------------------
Review API error: Google Hotels hasn't returned any results for this query.
No reviews found.
Hotel information saved.
Dataset saved: 502 records | 20 unique hotels
SUCCESS: Welcome to Pinterest Inspired Villa in center of the city.
New hotels collected: 20/100
Total unique hotels now: 20
Total records now: 502


/tmp/ipykernel_1058/801334368.py:1332: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(




SEARCHING: MI Road Jaipur
NEW HOTEL PROGRESS: 20/100
Hotels returned: 20

------------------------------------------------------------
NEW HOTEL 21/100
Hotel: Fairfield by Marriott Jaipur
Hotel ID: ChkIjo6Ouq_Dv5skGg0vZy8xMXN2cjl4emp5EAE
Area: MI Road Jaipur
------------------------------------------------------------
Reviews collected: 30
Dataset saved: 532 records | 21 unique hotels
SUCCESS: Fairfield by Marriott Jaipur
New hotels collected: 21/100
Total unique hotels now: 21
Total records now: 532

------------------------------------------------------------
NEW HOTEL 22/100
Hotel: ITC Rajputana, a Luxury Collection Hotel, Jaipur
Hotel ID: ChcIxvn2z9fmz8coGgsvZy8xdGR2bjQzdBAB
Area: MI Road Jaipur
------------------------------------------------------------
Reviews collected: 30
Dataset saved: 562 records | 22 unique hotels
SUCCESS: ITC Rajputana, a Luxury Collection Hotel, Jaipur
New hotels collected: 22/100
Total unique hotels now: 22
Total records now: 562
SKIP EXISTING: Hotel I

...



SEARCHING: Ashok Nagar Jaipur
NEW HOTEL PROGRESS: 36/100
Hotels returned: 20
SKIP EXISTING: Blue Beds Jaipur

------------------------------------------------------------
NEW HOTEL 37/100
Hotel: Hotel Royal Primes
Hotel ID: ChoI8KWF_dLX5ciUARoNL2cvMTF5aGtiMHN3ZBAB
Area: Ashok Nagar Jaipur
------------------------------------------------------------
Reviews collected: 30
Dataset saved: 938 records | 37 unique hotels
SUCCESS: Hotel Royal Primes
New hotels collected: 37/100
Total unique hotels now: 37
Total records now: 938

------------------------------------------------------------
NEW HOTEL 38/100
Hotel: Hotel Kalyan
Hotel ID: ChgIv_bomqi8iKrBARoLL2cvMXY3NXE5ZjIQAQ
Area: Ashok Nagar Jaipur
------------------------------------------------------------
Reviews collected: 30
Dataset saved: 968 records | 38 unique hotels
SUCCESS: Hotel Kalyan
New hotels collected: 38/100
Total unique hotels now: 38
Total records now: 968

------------------------------------------------------------
NEW H

/tmp/ipykernel_1058/801334368.py:1332: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(




SEARCHING: Panch Batti Jaipur
NEW HOTEL PROGRESS: 48/100
Hotels returned: 20
SKIP EXISTING: Shaan hotel
SKIP EXISTING: Hotel Jaisingh Palace
SKIP EXISTING: Hotel Silver Stay 04
SKIP EXISTING: Hotel Royal Primes

------------------------------------------------------------
NEW HOTEL 49/100
Hotel: Hotel Rangoli
Hotel ID: ChgInqX4nd6KvJDIARoLL2cvMXRkbjJoZjIQAQ
Area: Panch Batti Jaipur
------------------------------------------------------------
Reviews collected: 30
Dataset saved: 1233 records | 49 unique hotels
SUCCESS: Hotel Rangoli
New hotels collected: 49/100
Total unique hotels now: 49
Total records now: 1233

------------------------------------------------------------
NEW HOTEL 50/100
Hotel: Jaipur The Lodge B&B
Hotel ID: ChoIk4iO-J_Pu9rbARoNL2cvMTFjMnEwNXMybRAB
Area: Panch Batti Jaipur
------------------------------------------------------------
Reviews collected: 30
Dataset saved: 1263 records | 50 unique hotels
SUCCESS: Jaipur The Lodge B&B
New hotels collected: 50/100
Total u

/tmp/ipykernel_1058/801334368.py:1332: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(




SEARCHING: Civil Lines Jaipur
NEW HOTEL PROGRESS: 63/100
Hotels returned: 20

------------------------------------------------------------
NEW HOTEL 64/100
Hotel: Hari Mahal Palace by Pachar Group
Hotel ID: ChcI0emEysKxmIN4GgsvZy8xdGg1NWQ2ZxAB
Area: Civil Lines Jaipur
------------------------------------------------------------
Reviews collected: 30
Dataset saved: 1624 records | 64 unique hotels
SUCCESS: Hari Mahal Palace by Pachar Group
New hotels collected: 64/100
Total unique hotels now: 64
Total records now: 1624

------------------------------------------------------------
NEW HOTEL 65/100
Hotel: Hotel Shahar Palace
Hotel ID: ChcI8433pPjqpeI-GgsvZy8xdGYwMHdweBAB
Area: Civil Lines Jaipur
------------------------------------------------------------
Reviews collected: 30
Dataset saved: 1654 records | 65 unique hotels
SUCCESS: Hotel Shahar Palace
New hotels collected: 65/100
Total unique hotels now: 65
Total records now: 1654
SKIP EXISTING: goSTOPS Jaipur, Civil Lines

-------------

In [ ]:
from google.colab import files

files.download("jaipur_hotel_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import requests
import pandas as pd
import getpass
import time
import os

# ============================================================
# 1. CONFIGURATION
# ============================================================

API_KEY = getpass.getpass("Enter your SerpApi API key: ")

CITY = "Hyderabad"
STATE = "Telangana"
COUNTRY_CODE = "in"

CHECK_IN = "2026-10-15"
CHECK_OUT = "2026-10-16"

# IMPORTANT:
# We want ADDITIONAL 100 hotels, not total 100 hotels.
ADDITIONAL_HOTELS = 100

MAX_REVIEW_PAGES = 3

OUTPUT_FILE = "hyderabad_hotel_dataset.csv"

BASE_URL = "https://serpapi.com/search.json"


# ============================================================
# 2. SEARCH AREAS
# ============================================================

AREAS = [
    "Central Hyderabad",
    "HITEC City",
    "Gachibowli",
    "Madhapur",
    "Banjara Hills",
    "Jubilee Hills",
    "Secunderabad",
    "Begumpet",
    "Kondapur",
    "Shamshabad",
    "Abids",
    "Ameerpet",
    "Kukatpally",
    "Miyapur",
    "LB Nagar",
    "Manikonda",
    "Nampally",
    "Mehdipatnam",
    "Uppal",
    "Financial District",
    "Raidurg",
    "Koti",
    "Somajiguda",
    "Tolichowki",
    "Dilsukhnagar",
    "Kachiguda",
    "Charminar",
    "Lakdikapul",
    "Himayatnagar",
    "Attapur",
    "Bachupally",
    "Kompally",
    "Malkajgiri",
    "KPHB",
    "Gachibowli Financial District",
    "Kokapet",
    "Nanakramguda",
    "Chandanagar",
    "Sainikpuri",
    "Tarnaka",
    "Habsiguda",
    "Masab Tank",
    "Rajendra Nagar",
    "Shamshabad Airport",
    "Patancheru",
    "Boduppal",
    "Nagole",
    "ECIL",
    "Kondapur Botanical Garden"
]


# ============================================================
# 3. LOAD EXISTING CSV
# ============================================================

if os.path.exists(OUTPUT_FILE):

    existing_df = pd.read_csv(OUTPUT_FILE)

    print("\nExisting CSV found.")
    print("Existing rows:", len(existing_df))

    if "hotel_id" in existing_df.columns:
        existing_hotel_ids = set(
            existing_df["hotel_id"]
            .dropna()
            .astype(str)
            .str.strip()
        )
    else:
        existing_hotel_ids = set()

else:

    print("\nNo existing CSV found.")
    existing_df = pd.DataFrame()
    existing_hotel_ids = set()


print("Existing unique hotels:", len(existing_hotel_ids))


# ============================================================
# 4. HELPER FUNCTION
# ============================================================

def safe_request(params):

    params = params.copy()
    params["api_key"] = API_KEY

    try:

        response = requests.get(
            BASE_URL,
            params=params,
            timeout=60
        )

        if response.status_code != 200:

            print("API error:", response.status_code)
            print(response.text[:300])

            return {}

        data = response.json()

        if "error" in data:

            print("API error:", data["error"])

            return {}

        return data

    except requests.RequestException as e:

        print("Request failed:", e)

        return {}


# ============================================================
# 5. EXTRACT PRICE
# ============================================================

def extract_price(hotel):

    rate = hotel.get("rate_per_night", {})

    if isinstance(rate, dict):

        return (
            rate.get("extracted_lowest")
            or rate.get("extracted_before_taxes_fees")
            or rate.get("lowest")
            or rate.get("before_taxes_fees")
        )

    return rate if isinstance(rate, (int, float)) else None


# ============================================================
# 6. EXTRACT AMENITIES
# ============================================================

def extract_amenities(hotel):

    amenities = hotel.get("amenities", [])

    if isinstance(amenities, list):

        names = []

        for item in amenities:

            if isinstance(item, dict):

                name = (
                    item.get("name")
                    or item.get("title")
                )

                if name:
                    names.append(str(name))

            else:

                names.append(str(item))

        return " | ".join(names)

    return str(amenities) if amenities else None


# ============================================================
# 7. EXTRACT LOCATION
# ============================================================

def extract_location(hotel):

    address = hotel.get("address")

    if address:
        return address

    gps = hotel.get("gps_coordinates", {})

    if isinstance(gps, dict):

        lat = gps.get("latitude")
        lon = gps.get("longitude")

        if lat is not None and lon is not None:

            return f"Latitude: {lat}, Longitude: {lon}"

    return None


# ============================================================
# 8. EXTRACT REVIEW TEXT
# ============================================================

def extract_review_text(review):

    for key in ("snippet", "text"):

        value = review.get(key)

        if isinstance(value, str) and value.strip():

            return value.strip()

    return None


# ============================================================
# 9. EXTRACT REVIEW RATING
# ============================================================

def extract_review_rating(review):

    rating = review.get("rating")

    if isinstance(rating, dict):

        return (
            rating.get("value")
            or rating.get("rating")
        )

    return rating


# ============================================================
# 10. CREATE HOTEL ROW
# ============================================================

def make_hotel_row(hotel, area):

    return {

        "hotel_name": hotel.get("name"),

        "hotel_id": hotel.get("property_token"),

        "city": CITY,

        "location": extract_location(hotel),

        "price_per_night": extract_price(hotel),

        "hotel_category": hotel.get("hotel_class"),

        "customer_rating": hotel.get("overall_rating"),

        "review_count": hotel.get("reviews"),

        "amenities": extract_amenities(hotel),

        "search_area": area
    }


# ============================================================
# 11. FIND 100 NEW HOTELS
# ============================================================

new_hotels_by_id = {}

print("\n============================================================")
print("SEARCHING FOR ADDITIONAL HOTELS")
print("============================================================")

for area in AREAS:

    if len(new_hotels_by_id) >= ADDITIONAL_HOTELS:
        break

    print(
        f"\nSearching: Hotels in {area}, {CITY}"
    )

    next_page_token = None

    while len(new_hotels_by_id) < ADDITIONAL_HOTELS:

        params = {

            "engine": "google_hotels",

            "q": f"Hotels in {area}, {CITY}",

            "gl": COUNTRY_CODE,

            "hl": "en",

            "currency": "INR",

            "check_in_date": CHECK_IN,

            "check_out_date": CHECK_OUT,

            "adults": 2
        }

        if next_page_token:

            params["next_page_token"] = next_page_token

        data = safe_request(params)

        if not data:
            break

        properties = data.get("properties", [])

        if not properties:
            break

        for hotel in properties:

            hotel_id = hotel.get("property_token")

            if not hotel_id:
                continue

            # ------------------------------------------------
            # IMPORTANT:
            # SKIP HOTELS ALREADY PRESENT IN CSV
            # ------------------------------------------------

            if hotel_id in existing_hotel_ids:
                continue

            # ------------------------------------------------
            # SKIP DUPLICATES FOUND DURING THIS NEW SCRAPE
            # ------------------------------------------------

            if hotel_id in new_hotels_by_id:
                continue

            new_hotels_by_id[hotel_id] = {

                **make_hotel_row(
                    hotel,
                    area
                ),

                "_raw_hotel": hotel
            }

            print(
                f"NEW HOTEL {len(new_hotels_by_id)}/"
                f"{ADDITIONAL_HOTELS}: "
                f"{hotel.get('name')}"
            )

            if len(new_hotels_by_id) >= ADDITIONAL_HOTELS:
                break

        # ----------------------------------------------------
        # PAGINATION
        # ----------------------------------------------------

        next_page_token = (
            data.get(
                "serpapi_pagination",
                {}
            ).get("next_page_token")
        )

        if not next_page_token:
            break

        time.sleep(1)

    print(
        "New unique hotels collected so far:",
        len(new_hotels_by_id)
    )


print("\n============================================================")
print("HOTEL SEARCH COMPLETE")
print("============================================================")

print(
    "Existing unique hotels:",
    len(existing_hotel_ids)
)

print(
    "Additional unique hotels found:",
    len(new_hotels_by_id)
)

print(
    "Expected total unique hotels:",
    len(existing_hotel_ids) +
    len(new_hotels_by_id)
)


# ============================================================
# 12. FETCH REVIEWS FOR NEW HOTELS
# ============================================================

all_new_rows = []

skipped_reviews = 0

hotel_items = list(
    new_hotels_by_id.values()
)


print("\n============================================================")
print("FETCHING REVIEWS FOR NEW HOTELS")
print("============================================================")


for hotel_num, hotel_data in enumerate(
    hotel_items,
    start=1
):

    hotel = hotel_data["_raw_hotel"]

    hotel_id = hotel.get("property_token")

    print(
        f"\n[{hotel_num}/{len(hotel_items)}] "
        f"{hotel.get('name')}"
    )

    hotel_details = {

        key: value

        for key, value in hotel_data.items()

        if key != "_raw_hotel"
    }

    next_review_token = None

    review_pages = 0

    valid_reviews = 0

    while review_pages < MAX_REVIEW_PAGES:

        params = {

            "engine":
                "google_hotels_reviews",

            "property_token":
                hotel_id,

            "hl":
                "en",

            "sort_by":
                2
        }

        if next_review_token:

            params["next_page_token"] = (
                next_review_token
            )

        review_data = safe_request(
            params
        )

        if not review_data:
            break

        reviews = review_data.get(
            "reviews",
            []
        )

        if not reviews:
            break

        review_pages += 1

        for review in reviews:

            review_text = extract_review_text(
                review
            )

            if not review_text:

                skipped_reviews += 1

                continue

            row = hotel_details.copy()

            row.update({

                "review_text":
                    review_text,

                "individual_review_rating":
                    extract_review_rating(
                        review
                    ),

                "review_date":
                    review.get("date"),

                "review_source":
                    review.get("source")
            })

            all_new_rows.append(row)

            valid_reviews += 1

        next_review_token = (
            review_data
            .get(
                "serpapi_pagination",
                {}
            )
            .get(
                "next_page_token"
            )
        )

        if not next_review_token:
            break

        time.sleep(1)

    print(
        "Reviews with text:",
        valid_reviews
    )

    time.sleep(1)


# ============================================================
# 13. CREATE NEW DATAFRAME
# ============================================================

columns = [

    "hotel_name",
    "hotel_id",
    "city",
    "location",
    "price_per_night",
    "hotel_category",
    "customer_rating",
    "review_count",
    "amenities",
    "review_text",
    "individual_review_rating",
    "review_date",
    "review_source",
    "search_area"
]


new_df = pd.DataFrame(
    all_new_rows,
    columns=columns
)


# ============================================================
# 14. REMOVE EMPTY REVIEW TEXT
# ============================================================

if not new_df.empty:

    new_df = new_df[
        new_df["review_text"].notna()
        &
        new_df["review_text"]
        .astype(str)
        .str.strip()
        .ne("")
    ].copy()

    new_df.drop_duplicates(
        inplace=True
    )


# ============================================================
# 15. APPEND TO EXISTING DATASET
# ============================================================

if not existing_df.empty:

    combined_df = pd.concat(
        [
            existing_df,
            new_df
        ],
        ignore_index=True
    )

else:

    combined_df = new_df.copy()


# ============================================================
# 16. FINAL DEDUPLICATION
# ============================================================

# Remove exact duplicate rows
combined_df.drop_duplicates(
    inplace=True
)


# Remove duplicate hotel-review combinations
# if review_id is not available, hotel_id + review_text
# is used.

if (
    "hotel_id" in combined_df.columns
    and
    "review_text" in combined_df.columns
):

    combined_df.drop_duplicates(
        subset=[
            "hotel_id",
            "review_text"
        ],
        inplace=True
    )


# ============================================================
# 17. SAVE FINAL CSV
# ============================================================

combined_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 18. FINAL SUMMARY
# ============================================================

final_unique_hotels = (
    combined_df["hotel_id"]
    .nunique()
    if not combined_df.empty
    else 0
)

print("\n")
print("============================================================")
print("FINAL SUMMARY")
print("============================================================")

print(
    "Existing unique hotels:",
    len(existing_hotel_ids)
)

print(
    "Additional hotels found:",
    len(new_hotels_by_id)
)

print(
    "Final unique hotels:",
    final_unique_hotels
)

print(
    "Final total rows:",
    len(combined_df)
)

print(
    "New review rows added:",
    len(new_df)
)

print(
    "Reviews skipped for empty text:",
    skipped_reviews
)

print(
    "CSV saved:",
    OUTPUT_FILE
)

print("============================================================")

display(
    combined_df.head(10)
)

Enter your SerpApi API key: ··········

Existing CSV found.
Existing rows: 1055
Existing unique hotels: 86

SEARCHING FOR ADDITIONAL HOTELS

Searching: Hotels in Central Hyderabad, Hyderabad
NEW HOTEL 1/100: Hotel Virat Inn
NEW HOTEL 2/100: Modern 2 BHK unit in the heart of Hyderabad city\n
NEW HOTEL 3/100: Hotel AA Grand
NEW HOTEL 4/100: ITC Kakatiya, a Luxury Collection Hotel, Hyderabad
NEW HOTEL 5/100: Hotel Savera Residency
NEW HOTEL 6/100: OYO Grandeur By orbit Hotels Near Chennai shopping mall
NEW HOTEL 7/100: Hotel Sitara Paradise - Ameerpet
NEW HOTEL 8/100: Taj Deccan, Hyderabad
NEW HOTEL 9/100: Via Log Inn
NEW HOTEL 10/100: Hotel Al Diyafa
NEW HOTEL 11/100: GOLDEN POINT HOTEL
NEW HOTEL 12/100: Hotel suncity
NEW HOTEL 13/100: Hotel Polo
NEW HOTEL 14/100: Metro City Hostel
NEW HOTEL 15/100: Hotel Sai Leela Residency
NEW HOTEL 16/100: SPOT ON Hotel Prince Residency
NEW HOTEL 17/100: SPOT ON 806901 Hotel Park9
NEW HOTEL 18/100: OYO Somajiguda
NEW HOTEL 19/100: OYO Flagship 78735 H

,hotel_name,hotel_id,city,location,price_per_night,hotel_category,customer_rating,review_count,amenities,review_text,individual_review_rating,review_date,review_source,search_area
0,RCC Hotels Elite - Begumpet,ChkIwfHNgbqXvqtfGg0vZy8xMXJtcDNzMWcwEAE,Hyderabad,"Latitude: 17.4318882, Longitude: 78.4573388",1466.0,NaN,4.9,2337,Breakfast ($) | Free Wi-Fi | Free parking | Ai...,Booked last minute after seeing some negative ...,5,2 weeks ago,Google,Central Hyderabad
1,RCC Hotels Elite - Begumpet,ChkIwfHNgbqXvqtfGg0vZy8xMXJtcDNzMWcwEAE,Hyderabad,"Latitude: 17.4318882, Longitude: 78.4573388",1466.0,NaN,4.9,2337,Breakfast ($) | Free Wi-Fi | Free parking | Ai...,Begumpet,5,2 weeks ago,Google,Central Hyderabad
2,RCC Hotels Elite - Begumpet,ChkIwfHNgbqXvqtfGg0vZy8xMXJtcDNzMWcwEAE,Hyderabad,"Latitude: 17.4318882, Longitude: 78.4573388",1466.0,NaN,4.9,2337,Breakfast ($) | Free Wi-Fi | Free parking | Ai...,Complete Disaster of a Stay. Avoid If You Valu...,1,a month ago,Google,Central Hyderabad
3,RCC Hotels Elite - Begumpet,ChkIwfHNgbqXvqtfGg0vZy8xMXJtcDNzMWcwEAE,Hyderabad,"Latitude: 17.4318882, Longitude: 78.4573388",1466.0,NaN,4.9,2337,Breakfast ($) | Free Wi-Fi | Free parking | Ai...,Good,5,a month ago,Google,Central Hyderabad
4,RCC Hotels Elite - Begumpet,ChkIwfHNgbqXvqtfGg0vZy8xMXJtcDNzMWcwEAE,Hyderabad,"Latitude: 17.4318882, Longitude: 78.4573388",1466.0,NaN,4.9,2337,Breakfast ($) | Free Wi-Fi | Free parking | Ai...,Food and acamidation good an quality hygnice s...,5,a month ago,Google,Central Hyderabad
5,RCC Hotels Elite - Begumpet,ChkIwfHNgbqXvqtfGg0vZy8xMXJtcDNzMWcwEAE,Hyderabad,"Latitude: 17.4318882, Longitude: 78.4573388",1466.0,NaN,4.9,2337,Breakfast ($) | Free Wi-Fi | Free parking | Ai...,Well maintained room's and all good,5,a month ago,Google,Central Hyderabad
6,RCC Hotels Elite - Begumpet,ChkIwfHNgbqXvqtfGg0vZy8xMXJtcDNzMWcwEAE,Hyderabad,"Latitude: 17.4318882, Longitude: 78.4573388",1466.0,NaN,4.9,2337,Breakfast ($) | Free Wi-Fi | Free parking | Ai...,Good location,5,a month ago,Google,Central Hyderabad
7,RCC Hotels Elite - Begumpet,ChkIwfHNgbqXvqtfGg0vZy8xMXJtcDNzMWcwEAE,Hyderabad,"Latitude: 17.4318882, Longitude: 78.4573388",1466.0,NaN,4.9,2337,Breakfast ($) | Free Wi-Fi | Free parking | Ai...,**Very Disappointing Experience – Not Recommen...,1,a month ago,Google,Central Hyderabad
8,RCC Hotels Elite - Begumpet,ChkIwfHNgbqXvqtfGg0vZy8xMXJtcDNzMWcwEAE,Hyderabad,"Latitude: 17.4318882, Longitude: 78.4573388",1466.0,NaN,4.9,2337,Breakfast ($) | Free Wi-Fi | Free parking | Ai...,Very Disappointing Stay\nThe room had a very b...,1,a month ago,Google,Central Hyderabad
9,RCC Hotels Elite - Begumpet,ChkIwfHNgbqXvqtfGg0vZy8xMXJtcDNzMWcwEAE,Hyderabad,"Latitude: 17.4318882, Longitude: 78.4573388",1466.0,NaN,4.9,2337,Breakfast ($) | Free Wi-Fi | Free parking | Ai...,Very Worst Hotel – Not Recommended\nVery disap...,1,a month ago,Google,Central Hyderabad


In [ ]:
from google.colab import files

files.download("hyderabad_hotel_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import requests
import pandas as pd
import getpass
import time
import os


# ============================================================
# 1. CONFIGURATION
# ============================================================

API_KEY = getpass.getpass("Enter your SerpApi API key: ")

CITY = "Delhi"
STATE = "Delhi"
COUNTRY_CODE = "in"

# Google Hotels requires stay dates
CHECK_IN = "2026-10-15"
CHECK_OUT = "2026-10-16"

# IMPORTANT:
# We want 100 ADDITIONAL hotels
ADDITIONAL_HOTELS = 100

MAX_REVIEW_PAGES = 3

OUTPUT_FILE = "delhi_hotel_dataset.csv"

BASE_URL = "https://serpapi.com/search.json"


# ============================================================
# 2. SEARCH AREAS
# ============================================================

AREAS = [

    "Central Delhi",
    "Connaught Place",
    "Karol Bagh",
    "Paharganj",
    "Aerocity",
    "Dwarka",
    "Rohini",
    "Saket",
    "Hauz Khas",
    "Vasant Kunj",
    "Greater Kailash",
    "South Extension",
    "Lajpat Nagar",
    "Nehru Place",
    "Chanakyapuri",
    "Kashmere Gate",
    "Civil Lines",
    "Rajouri Garden",
    "Janakpuri",
    "Pitampura",
    "Mayur Vihar",
    "Okhla",
    "Green Park",
    "Defence Colony",
    "Jasola",
    "Mahipalpur",
    "Laxmi Nagar",
    "ITO",
    "Pusa Road",
    "New Friends Colony",

    # Additional areas for better coverage
    "Shahdara",
    "Preet Vihar",
    "Anand Vihar",
    "Karkardooma",
    "Vivek Vihar",
    "Patparganj",
    "East Delhi",
    "South Delhi",
    "North Delhi",
    "West Delhi",
    "Old Delhi",
    "Daryaganj",
    "Chandni Chowk",
    "Sadar Bazaar",
    "Jama Masjid",
    "Jangpura",
    "Kailash Colony",
    "Kalkaji",
    "CR Park",
    "Govindpuri",
    "Greater Kailash 1",
    "Greater Kailash 2",
    "Malviya Nagar",
    "Chirag Delhi",
    "Panchsheel Park",
    "Safdarjung Enclave",
    "Vasant Vihar",
    "R K Puram",
    "Munirka",
    "Ber Sarai",
    "Mehrauli",
    "Chhatarpur",
    "Sultanpur",
    "Aya Nagar",
    "Sarita Vihar",
    "Madanpur Khadar",
    "Kalindi Kunj",
    "Tughlakabad",
    "Jasola Vihar",
    "Govindpuri Extension",
    "Nangloi",
    "Uttam Nagar",
    "Tilak Nagar",
    "Subhash Nagar",
    "Tagore Garden",
    "Punjabi Bagh",
    "Paschim Vihar",
    "Rajendra Place",
    "Shadipur",
    "Patel Nagar",
    "Shalimar Bagh",
    "Model Town",
    "Kamla Nagar",
    "Mukherjee Nagar",
    "GTB Nagar",
    "Vijay Nagar",
    "Ashok Vihar",
    "Wazirpur",
    "Azadpur",
    "Kashmere Gate ISBT",
    "Indira Gandhi International Airport",
    "Delhi Airport",
    "Airport Road"
]


# ============================================================
# 3. LOAD EXISTING CSV
# ============================================================

if os.path.exists(OUTPUT_FILE):

    existing_df = pd.read_csv(OUTPUT_FILE)

    print("\nExisting Delhi CSV found.")
    print(
        "Existing rows:",
        len(existing_df)
    )

    if "hotel_id" in existing_df.columns:

        existing_hotel_ids = set(
            existing_df["hotel_id"]
            .dropna()
            .astype(str)
            .str.strip()
        )

    else:

        existing_hotel_ids = set()

else:

    print("\nNo existing Delhi CSV found.")

    existing_df = pd.DataFrame()

    existing_hotel_ids = set()


print(
    "Existing unique hotels:",
    len(existing_hotel_ids)
)


# ============================================================
# 4. SAFE API REQUEST
# ============================================================

def safe_request(params):

    params = params.copy()

    params["api_key"] = API_KEY

    try:

        response = requests.get(
            BASE_URL,
            params=params,
            timeout=60
        )

        if response.status_code != 200:

            print(
                "API error:",
                response.status_code
            )

            print(
                response.text[:300]
            )

            return {}

        data = response.json()

        if "error" in data:

            print(
                "API error:",
                data["error"]
            )

            return {}

        return data

    except requests.RequestException as e:

        print(
            "Request failed:",
            e
        )

        return {}


# ============================================================
# 5. EXTRACT PRICE
# ============================================================

def extract_price(hotel):

    rate = hotel.get(
        "rate_per_night",
        {}
    )

    if isinstance(rate, dict):

        return (

            rate.get("extracted_lowest")

            or rate.get(
                "extracted_before_taxes_fees"
            )

            or rate.get("lowest")

            or rate.get(
                "before_taxes_fees"
            )
        )

    return (
        rate
        if isinstance(
            rate,
            (int, float)
        )
        else None
    )


# ============================================================
# 6. EXTRACT AMENITIES
# ============================================================

def extract_amenities(hotel):

    amenities = hotel.get(
        "amenities",
        []
    )

    if isinstance(
        amenities,
        list
    ):

        names = []

        for item in amenities:

            if isinstance(
                item,
                dict
            ):

                name = (
                    item.get("name")
                    or
                    item.get("title")
                )

                if name:

                    names.append(
                        str(name)
                    )

            else:

                names.append(
                    str(item)
                )

        return " | ".join(names)

    return (
        str(amenities)
        if amenities
        else None
    )


# ============================================================
# 7. EXTRACT LOCATION
# ============================================================

def extract_location(hotel):

    address = hotel.get(
        "address"
    )

    if address:

        return address

    gps = hotel.get(
        "gps_coordinates",
        {}
    )

    if isinstance(
        gps,
        dict
    ):

        lat = gps.get(
            "latitude"
        )

        lon = gps.get(
            "longitude"
        )

        if (
            lat is not None
            and
            lon is not None
        ):

            return (
                f"Latitude: {lat}, "
                f"Longitude: {lon}"
            )

    return None


# ============================================================
# 8. EXTRACT REVIEW TEXT
# ============================================================

def extract_review_text(review):

    for key in (
        "snippet",
        "text"
    ):

        value = review.get(
            key
        )

        if (
            isinstance(
                value,
                str
            )
            and
            value.strip()
        ):

            return value.strip()

    return None


# ============================================================
# 9. EXTRACT REVIEW RATING
# ============================================================

def extract_review_rating(review):

    rating = review.get(
        "rating"
    )

    if isinstance(
        rating,
        dict
    ):

        return (
            rating.get("value")
            or
            rating.get("rating")
        )

    return rating


# ============================================================
# 10. CREATE HOTEL ROW
# ============================================================

def make_hotel_row(
    hotel,
    area
):

    return {

        "hotel_name":
            hotel.get("name"),

        "hotel_id":
            hotel.get(
                "property_token"
            ),

        "city":
            CITY,

        "location":
            extract_location(
                hotel
            ),

        "price_per_night":
            extract_price(
                hotel
            ),

        "hotel_category":
            hotel.get(
                "hotel_class"
            ),

        "customer_rating":
            hotel.get(
                "overall_rating"
            ),

        "review_count":
            hotel.get(
                "reviews"
            ),

        "amenities":
            extract_amenities(
                hotel
            ),

        "search_area":
            area
    }


# ============================================================
# 11. SEARCH FOR 100 NEW HOTELS
# ============================================================

new_hotels_by_id = {}

print("\n")
print("=" * 60)
print("SEARCHING FOR 100 ADDITIONAL DELHI HOTELS")
print("=" * 60)

for area in AREAS:

    # Stop once 100 NEW hotels are found
    if (
        len(new_hotels_by_id)
        >= ADDITIONAL_HOTELS
    ):

        break

    print(
        f"\nSearching: Hotels in "
        f"{area}, {CITY}"
    )

    next_page_token = None

    while (
        len(new_hotels_by_id)
        < ADDITIONAL_HOTELS
    ):

        params = {

            "engine":
                "google_hotels",

            "q":
                f"Hotels in {area}, {CITY}",

            "gl":
                COUNTRY_CODE,

            "hl":
                "en",

            "currency":
                "INR",

            "check_in_date":
                CHECK_IN,

            "check_out_date":
                CHECK_OUT,

            "adults":
                2
        }

        # Add pagination token
        if next_page_token:

            params[
                "next_page_token"
            ] = next_page_token

        data = safe_request(
            params
        )

        if not data:

            break

        properties = data.get(
            "properties",
            []
        )

        if not properties:

            break

        for hotel in properties:

            hotel_id = hotel.get(
                "property_token"
            )

            if not hotel_id:

                continue

            # ==================================================
            # VERY IMPORTANT:
            # SKIP HOTELS ALREADY IN EXISTING CSV
            # ==================================================

            if (
                str(hotel_id)
                in existing_hotel_ids
            ):

                continue

            # ==================================================
            # SKIP DUPLICATES FROM CURRENT SCRAPE
            # ==================================================

            if (
                hotel_id
                in new_hotels_by_id
            ):

                continue

            # ==================================================
            # ADD NEW HOTEL
            # ==================================================

            new_hotels_by_id[
                hotel_id
            ] = {

                **make_hotel_row(
                    hotel,
                    area
                ),

                "_raw_hotel":
                    hotel
            }

            print(
                f"NEW HOTEL "
                f"{len(new_hotels_by_id)}/"
                f"{ADDITIONAL_HOTELS}: "
                f"{hotel.get('name')}"
            )

            if (
                len(new_hotels_by_id)
                >= ADDITIONAL_HOTELS
            ):

                break

        # ======================================================
        # GET NEXT HOTEL SEARCH PAGE
        # ======================================================

        next_page_token = (

            data.get(
                "serpapi_pagination",
                {}
            ).get(
                "next_page_token"
            )
        )

        if not next_page_token:

            break

        time.sleep(1)

    print(
        "New unique hotels collected:",
        len(new_hotels_by_id)
    )


# ============================================================
# 12. HOTEL SEARCH SUMMARY
# ============================================================

print("\n")
print("=" * 60)
print("HOTEL SEARCH COMPLETE")
print("=" * 60)

print(
    "Existing unique hotels:",
    len(existing_hotel_ids)
)

print(
    "Additional unique hotels found:",
    len(new_hotels_by_id)
)

print(
    "Expected final unique hotels:",
    len(existing_hotel_ids)
    +
    len(new_hotels_by_id)
)


# ============================================================
# 13. FETCH REVIEWS FOR NEW HOTELS
# ============================================================

all_new_rows = []

skipped_reviews = 0

hotel_items = list(
    new_hotels_by_id.values()
)

print("\n")
print("=" * 60)
print("FETCHING REVIEWS FOR NEW HOTELS")
print("=" * 60)


for hotel_num, hotel_data in enumerate(
    hotel_items,
    start=1
):

    hotel = hotel_data[
        "_raw_hotel"
    ]

    hotel_id = hotel.get(
        "property_token"
    )

    print(
        f"\n[{hotel_num}/"
        f"{len(hotel_items)}] "
        f"{hotel.get('name')}"
    )

    hotel_details = {

        key: value

        for key, value
        in hotel_data.items()

        if key != "_raw_hotel"
    }

    next_review_token = None

    review_pages = 0

    valid_reviews = 0

    while (
        review_pages
        < MAX_REVIEW_PAGES
    ):

        params = {

            "engine":
                "google_hotels_reviews",

            "property_token":
                hotel_id,

            "hl":
                "en",

            # Most recent reviews
            "sort_by":
                2
        }

        if next_review_token:

            params[
                "next_page_token"
            ] = next_review_token

        review_data = safe_request(
            params
        )

        if not review_data:

            break

        reviews = review_data.get(
            "reviews",
            []
        )

        if not reviews:

            break

        review_pages += 1

        for review in reviews:

            review_text = (
                extract_review_text(
                    review
                )
            )

            if not review_text:

                skipped_reviews += 1

                continue

            row = hotel_details.copy()

            row.update({

                "review_text":
                    review_text,

                "individual_review_rating":
                    extract_review_rating(
                        review
                    ),

                "review_date":
                    review.get(
                        "date"
                    ),

                "review_source":
                    review.get(
                        "source"
                    )
            })

            all_new_rows.append(
                row
            )

            valid_reviews += 1

        # ======================================================
        # NEXT REVIEW PAGE
        # ======================================================

        next_review_token = (

            review_data.get(
                "serpapi_pagination",
                {}
            ).get(
                "next_page_token"
            )
        )

        if not next_review_token:

            break

        time.sleep(1)

    print(
        "Reviews with text:",
        valid_reviews
    )

    time.sleep(1)


# ============================================================
# 14. CREATE NEW DATAFRAME
# ============================================================

columns = [

    "hotel_name",
    "hotel_id",
    "city",
    "location",
    "price_per_night",
    "hotel_category",
    "customer_rating",
    "review_count",
    "amenities",
    "review_text",
    "individual_review_rating",
    "review_date",
    "review_source",
    "search_area"
]


new_df = pd.DataFrame(
    all_new_rows,
    columns=columns
)


# ============================================================
# 15. REMOVE EMPTY REVIEW TEXT
# ============================================================

if not new_df.empty:

    new_df = new_df[
        new_df[
            "review_text"
        ].notna()
        &
        new_df[
            "review_text"
        ]
        .astype(str)
        .str.strip()
        .ne("")
    ].copy()

    new_df.drop_duplicates(
        inplace=True
    )


# ============================================================
# 16. APPEND NEW DATA TO EXISTING CSV
# ============================================================

if not existing_df.empty:

    combined_df = pd.concat(
        [
            existing_df,
            new_df
        ],
        ignore_index=True
    )

else:

    combined_df = new_df.copy()


# ============================================================
# 17. FINAL DEDUPLICATION
# ============================================================

combined_df.drop_duplicates(
    inplace=True
)


# Remove duplicate hotel + review-text combinations
if (
    "hotel_id"
    in combined_df.columns
    and
    "review_text"
    in combined_df.columns
):

    combined_df.drop_duplicates(
        subset=[
            "hotel_id",
            "review_text"
        ],
        inplace=True
    )


# ============================================================
# 18. SAVE FINAL CSV
# ============================================================

combined_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 19. FINAL SUMMARY
# ============================================================

final_unique_hotels = (

    combined_df[
        "hotel_id"
    ].nunique()

    if not combined_df.empty

    else 0
)


print("\n")
print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)

print(
    "Existing unique hotels:",
    len(existing_hotel_ids)
)

print(
    "Additional hotels found:",
    len(new_hotels_by_id)
)

print(
    "Final unique hotels:",
    final_unique_hotels
)

print(
    "Final total rows:",
    len(combined_df)
)

print(
    "New review rows added:",
    len(new_df)
)

print(
    "Reviews skipped for empty text:",
    skipped_reviews
)

print(
    "CSV saved:",
    OUTPUT_FILE
)

print("=" * 60)


display(
    combined_df.head(10)
)

Enter your SerpApi API key: ··········

Existing Delhi CSV found.
Existing rows: 1345
Existing unique hotels: 83


SEARCHING FOR 100 ADDITIONAL DELHI HOTELS

Searching: Hotels in Central Delhi, Delhi
NEW HOTEL 1/100: Entire 1BHK Penthouse with Private Terrace 4th Flr
NEW HOTEL 2/100: The Lalit New Delhi
NEW HOTEL 3/100: The Metropolitan Hotel & Spa
NEW HOTEL 4/100: Ahuja Residency Sunder Nagar
NEW HOTEL 5/100: Hotel Connaught Royale, Delhi
NEW HOTEL 6/100: Radisson Blu Marina Hotel, Delhi Connaught Place
NEW HOTEL 7/100: Hotel Emporio Dx Paharganj New Delhi
NEW HOTEL 8/100: Hotel Shivam international
NEW HOTEL 9/100: Homlee Heritage Flat with kitchen in Central Delhi
NEW HOTEL 10/100: Hotel All In Homes
NEW HOTEL 11/100: The Twenties Hotel Kamla Nagar
NEW HOTEL 12/100: Hotel Aman Continental Delhi
NEW HOTEL 13/100: BnBBuddy Executive Studio Apt in West Delhi 720
NEW HOTEL 14/100: Royale Avenue BnB
NEW HOTEL 15/100: Hotel O Delhi Continental
NEW HOTEL 16/100: Nina Kochhar's Homestay - D

,hotel_name,hotel_id,city,location,price_per_night,hotel_category,customer_rating,review_count,amenities,review_text,individual_review_rating,review_date,review_source,search_area
0,Jyoti Mahal A Royal Heritage Hotel -Close to C...,ChgIuvLVx_qd7vgwGgwvZy8xcHR3aDhnbTIQAQ,Delhi,"Latitude: 28.642041, Longitude: 77.2117176",4480.0,3-star hotel,3.7,1548.0,Free breakfast | Free Wi-Fi | Free parking | A...,I wouldn't recommend the most basic and cheape...,4.0,a week ago,Google,Central Delhi
1,Jyoti Mahal A Royal Heritage Hotel -Close to C...,ChgIuvLVx_qd7vgwGgwvZy8xcHR3aDhnbTIQAQ,Delhi,"Latitude: 28.642041, Longitude: 77.2117176",4480.0,3-star hotel,3.7,1548.0,Free breakfast | Free Wi-Fi | Free parking | A...,very beautiful! peaceful! and maahi is super s...,5.0,2 weeks ago,Google,Central Delhi
2,Jyoti Mahal A Royal Heritage Hotel -Close to C...,ChgIuvLVx_qd7vgwGgwvZy8xcHR3aDhnbTIQAQ,Delhi,"Latitude: 28.642041, Longitude: 77.2117176",4480.0,3-star hotel,3.7,1548.0,Free breakfast | Free Wi-Fi | Free parking | A...,Ranju is very good staff and help phul men,5.0,2 weeks ago,Google,Central Delhi
3,Jyoti Mahal A Royal Heritage Hotel -Close to C...,ChgIuvLVx_qd7vgwGgwvZy8xcHR3aDhnbTIQAQ,Delhi,"Latitude: 28.642041, Longitude: 77.2117176",4480.0,3-star hotel,3.7,1548.0,Free breakfast | Free Wi-Fi | Free parking | A...,"This hotel is definitely not as described, the...",1.0,a month ago,Tripadvisor,Central Delhi
4,Jyoti Mahal A Royal Heritage Hotel -Close to C...,ChgIuvLVx_qd7vgwGgwvZy8xcHR3aDhnbTIQAQ,Delhi,"Latitude: 28.642041, Longitude: 77.2117176",4480.0,3-star hotel,3.7,1548.0,Free breakfast | Free Wi-Fi | Free parking | A...,(Translated by Google) A wonderful place in th...,5.0,Edited a month ago,Google,Central Delhi
5,Jyoti Mahal A Royal Heritage Hotel -Close to C...,ChgIuvLVx_qd7vgwGgwvZy8xcHR3aDhnbTIQAQ,Delhi,"Latitude: 28.642041, Longitude: 77.2117176",4480.0,3-star hotel,3.7,1548.0,Free breakfast | Free Wi-Fi | Free parking | A...,(Translated by Google) My experience at Jyoti ...,5.0,a month ago,Google,Central Delhi
6,Jyoti Mahal A Royal Heritage Hotel -Close to C...,ChgIuvLVx_qd7vgwGgwvZy8xcHR3aDhnbTIQAQ,Delhi,"Latitude: 28.642041, Longitude: 77.2117176",4480.0,3-star hotel,3.7,1548.0,Free breakfast | Free Wi-Fi | Free parking | A...,"(Translated by Google) Average hotel, the best...",5.0,a month ago,Google,Central Delhi
7,Jyoti Mahal A Royal Heritage Hotel -Close to C...,ChgIuvLVx_qd7vgwGgwvZy8xcHR3aDhnbTIQAQ,Delhi,"Latitude: 28.642041, Longitude: 77.2117176",4480.0,3-star hotel,3.7,1548.0,Free breakfast | Free Wi-Fi | Free parking | A...,The location felt unsafe and was not a place w...,3.0,a month ago,Google,Central Delhi
8,Jyoti Mahal A Royal Heritage Hotel -Close to C...,ChgIuvLVx_qd7vgwGgwvZy8xcHR3aDhnbTIQAQ,Delhi,"Latitude: 28.642041, Longitude: 77.2117176",4480.0,3-star hotel,3.7,1548.0,Free breakfast | Free Wi-Fi | Free parking | A...,Good experience,5.0,2 months ago,Google,Central Delhi
9,Jyoti Mahal A Royal Heritage Hotel -Close to C...,ChgIuvLVx_qd7vgwGgwvZy8xcHR3aDhnbTIQAQ,Delhi,"Latitude: 28.642041, Longitude: 77.2117176",4480.0,3-star hotel,3.7,1548.0,Free breakfast | Free Wi-Fi | Free parking | A...,I really liked how they have presented their h...,5.0,2 months ago,Google,Central Delhi


In [ ]:
from google.colab import files

files.download("delhi_hotel_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>